# 08. Event Business Monte Carlo Simulator  
# 08. 赛事业务蒙特卡洛模拟器

**Pipeline position / 主线位置:** validated Bears event pattern -> synthetic two-day trip scenarios -> fleet, pricing, and profit-risk comparison.  
**流程位置：** 已验证的 Bears 赛事规律 -> 连续两天合成订单场景 -> 车队、定价和利润风险对比。

The simulator learns normal traffic, event effects, route shares, fares, tips, and duration from real completed trips. It then generates 20,000 random two-day scenarios and runs four fleet sizes with three pricing policies. The output is a distribution of outcomes, not one fixed guess. Contribution profit follows the stated scenario assumptions and is not audited company net profit.  
模拟器从真实完成订单中学习普通日客流、赛事影响、路线占比、车费、小费和时长，再生成 20,000 个随机的连续两天场景，并测试四种车队规模和三种定价策略。输出是一组结果分布，不是一个固定猜测。贡献利润按场景假设计算，不等同于企业审计后的净利润。

**Inputs / 输入:** Bears event pattern and trip economics from MatrixOne.  
**Outputs / 输出:** validated generator metrics, 20,000-run demand intervals, shortage risk and policy profit distributions.

In [ ]:
# Cell 1 - Install dependencies / 安装依赖 / Run once if packages are missing. / 缺少依赖时运行一次。
%pip install -q pandas numpy matplotlib seaborn pymysql h3 scikit-learn folium geopandas shapely contextily tqdm certifi

In [ ]:
from getpass import getpass
# Cell 2 — Imports and project configuration / 导入包和项目配置

from pathlib import Path
from IPython.display import display
import hashlib
import json
import math
import os
import warnings

import certifi
import contextily as cx
import folium
import geopandas as gpd
import h3
import matplotlib.font_manager as font_manager
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymysql
import seaborn as sns
from shapely.geometry import Point, Polygon
from tqdm.auto import tqdm

# Use certifi for online map tiles. / 在线地图瓦片使用 certifi 证书。
os.environ.setdefault("SSL_CERT_FILE", certifi.where())
os.environ.setdefault("REQUESTS_CA_BUNDLE", certifi.where())

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 240)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.unicode_minus"] = False

# Chinese font / 中文字体
font_candidates = [
    "PingFang SC", "Arial Unicode MS", "Heiti TC", "STHeiti",
    "Songti SC", "Noto Sans CJK SC",
]
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
CHINESE_FONT = next(
    (font for font in font_candidates if font in available_fonts),
    None,
)
if CHINESE_FONT:
    plt.rcParams["font.sans-serif"] = [CHINESE_FONT, "DejaVu Sans"]

# Paths / 路径
PROJECT_DIR = Path(
    os.getenv("CHICAGO_TNP_PROJECT_DIR", str(Path.cwd()))
).expanduser().resolve()
OUTPUT_DIR = PROJECT_DIR / "notebook_outputs_event_business_simulator"
CACHE_DIR = OUTPUT_DIR / "cache"
MAP_DIR = OUTPUT_DIR / "maps"
MONTE_CARLO_ROOT = OUTPUT_DIR / "monte_carlo_parts"
TRIP_EXPORT_DIR = OUTPUT_DIR / "representative_synthetic_trips"
for folder in [OUTPUT_DIR, CACHE_DIR, MAP_DIR, MONTE_CARLO_ROOT, TRIP_EXPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# MatrixOne / MatrixOne 配置
MO_HOST = os.getenv("MATRIXONE_HOST", "127.0.0.1")
MO_PORT = int(os.getenv("MATRIXONE_PORT", "6001"))
MO_USER = os.getenv("MATRIXONE_USER", "root")
MO_PASSWORD = os.getenv("MATRIXONE_PASSWORD") or getpass("MatrixOne password / MatrixOne 密码: ")
MO_DATABASE = os.getenv("MATRIXONE_DATABASE", "chicago_tnp")
ANALYSIS_TABLE = os.getenv(
    "MATRIXONE_ANALYSIS_TABLE", "unified_trips_h3_res9"
)

# Soldier Field and H3 / Soldier Field 与 H3
STADIUM_NAME = "Soldier Field"
STADIUM_LAT = 41.862313
STADIUM_LON = -87.616688
H3_RESOLUTION = 9
PREVALIDATED_CORE_H3 = [
    "892664c1b0bffff",
    "892664c1b47ffff",
]
SIMULATOR_VERSION = "empirical-event-tail-v1"
N_SELECTED_ACTIVE_H3 = 6
MAX_OBSERVED_H3_DISTANCE_KM = 2.0
MIN_H3_TOTAL_ACTIVITY = 1_000
CANDIDATE_GRID_RING = 8
GEOMETRIC_CONTEXT_RING = 1
EVENT_STRENGTH_JITTER_LOG_SD = 0.05
EVENT_STRENGTH_BOUND_PADDING = 0.10

# Data windows / 数据时间范围
DATA_START = pd.Timestamp("2022-01-01 00:00:00")
TRAIN_END = pd.Timestamp("2024-01-01 00:00:00")
DATA_END = pd.Timestamp("2025-01-01 00:00:00")
CALIBRATION_END = pd.Timestamp("2024-08-01 00:00:00")
RELATIVE_HOURS = list(range(-6, 9))
EVENT_EXCLUSION_HOURS = list(range(-8, 11))
MIN_NORMAL_CONTROLS = 8

# Two-day simulation / 连续两天模拟
SIMULATION_KICKOFF = pd.Timestamp("2025-09-14 12:00:00")
SIMULATION_HOURS = 48
N_SIMULATIONS = int(os.getenv("TNP_MONTE_CARLO_RUNS", "20000"))
BATCH_SIZE = int(os.getenv("TNP_MONTE_CARLO_BATCH_SIZE", "500"))
RANDOM_STATE = 42
RUN_FULL_MONTE_CARLO = True
FORCE_REFRESH_DATABASE_CACHE = False
FORCE_RERUN_MONTE_CARLO = False

# Explicit business assumptions / 明确的经营假设
PLATFORM_TAKE_RATE = 0.25
PLATFORM_VARIABLE_COST_PER_TRIP = 0.35
LOCAL_REBALANCE_SHARE = 0.30
MIN_LOCAL_RESERVE = 2

# Sample supply once per world and share it across policies. / 每个随机世界只抽一次供给，所有策略共用。
DROPOFF_AVAILABILITY_MEAN = 0.85
DROPOFF_AVAILABILITY_CONCENTRATION = 50.0
IDLE_VEHICLE_RETENTION_MEAN = 0.85
IDLE_VEHICLE_RETENTION_CONCENTRATION = 50.0
INITIAL_FLEET_LOG_SD = 0.12

FLEET_SCENARIOS = {
    "constrained": 0.35,
    "lean": 0.65,
    "standard": 1.00,
    "ample": 1.35,
}

PRICING_STRATEGIES = {
    "fixed_price": {
        "price_pressure_coefficient": 0.00,
        "max_price_multiplier": 1.00,
        "demand_elasticity": 0.00,
        "supply_response": 0.00,
        "driver_incentive_share": 0.00,
    },
    "balanced_surge": {
        "price_pressure_coefficient": 0.35,
        "max_price_multiplier": 1.50,
        "demand_elasticity": -0.20,
        "supply_response": 0.25,
        "driver_incentive_share": 0.12,
    },
    "aggressive_surge": {
        "price_pressure_coefficient": 0.65,
        "max_price_multiplier": 2.00,
        "demand_elasticity": -0.35,
        "supply_response": 0.45,
        "driver_incentive_share": 0.20,
    },
}

print("Output directory / 输出目录:", OUTPUT_DIR)
print("Monte Carlo runs / 蒙特卡洛次数:", f"{N_SIMULATIONS:,}")
print("Event day / 比赛日:", SIMULATION_KICKOFF.date())
print("Control day / 普通日:", (SIMULATION_KICKOFF + pd.Timedelta(days=1)).date())
print("Strategy combinations / 策略组合:", len(FLEET_SCENARIOS) * len(PRICING_STRATEGIES))

In [ ]:
# Cell 3 - H3 and MatrixOne helpers / H3 与 MatrixOne 辅助函数

def latlng_to_cell(lat, lng, resolution):
    """Convert coordinates to an H3 cell with h3-py v3/v4 support. / 将坐标转换为 H3，并兼容 h3-py v3/v4。"""
    # Support h3-py v3 and v4. / 兼容 h3-py v3 与 v4。
    if hasattr(h3, "latlng_to_cell"):
        return h3.latlng_to_cell(lat, lng, resolution)
    return h3.geo_to_h3(lat, lng, resolution)


def cell_to_latlng(cell):
    """Return the latitude and longitude of an H3 cell center. / 返回 H3 中心点的经纬度。"""
    # Return cell center as latitude and longitude. / 返回格子中心经纬度。
    if hasattr(h3, "cell_to_latlng"):
        return h3.cell_to_latlng(cell)
    return h3.h3_to_geo(cell)


def cell_to_boundary(cell):
    """Return the boundary coordinates of an H3 cell. / 返回 H3 边界坐标。"""
    # Return boundary points as (lat, lng). / 返回边界点 (纬度, 经度)。
    if hasattr(h3, "cell_to_boundary"):
        return list(h3.cell_to_boundary(cell))
    return list(h3.h3_to_geo_boundary(cell, geo_json=False))


def grid_disk(cell, radius):
    """Return the H3 cells within the requested grid distance. / 返回指定网格距离内的 H3。"""
    # Return H3 neighbors within a radius. / 返回指定圈数内的 H3 邻居。
    if hasattr(h3, "grid_disk"):
        return set(h3.grid_disk(cell, radius))
    return set(h3.k_ring(cell, radius))


def haversine_km(lat1, lon1, lat2, lon2):
    """Calculate great-circle distance in kilometers. / 计算两点间的球面距离（公里）。"""
    # Great-circle distance in kilometers. / 球面距离（公里）。
    radius = 6371.0088
    phi1, phi2 = np.radians([lat1, lat2])
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    value = (
        np.sin(dphi / 2) ** 2
        + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
    )
    return float(2 * radius * np.arcsin(np.sqrt(value)))


def connect_matrixone():
    """Open a MatrixOne connection with long-query timeouts. / 使用长查询超时设置连接 MatrixOne。"""
    # Open a MatrixOne connection. / 建立 MatrixOne 连接。
    return pymysql.connect(
        host=MO_HOST,
        port=MO_PORT,
        user=MO_USER,
        password=MO_PASSWORD,
        database=MO_DATABASE,
        charset="utf8mb4",
        autocommit=True,
        read_timeout=3600,
        write_timeout=3600,
    )


conn = connect_matrixone()


def query_df(sql, params=None):
    """Run SQL with one reconnect retry and return a DataFrame. / 执行 SQL，断线时重连一次，并返回 DataFrame。"""
    # Run SQL and reconnect after an idle disconnect. / 执行 SQL，断线时自动重连。
    global conn
    try:
        conn.ping()
    except Exception:
        try:
            conn.close()
        except Exception:
            pass
        conn = connect_matrixone()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return pd.read_sql_query(sql, conn, params=params)


def sql_string(value):
    """Escape one value for use as a SQL string literal. / 转义 SQL 字符串值。"""
    # Quote a SQL string safely. / 安全生成 SQL 字符串。
    return "'" + str(value).replace("'", "''") + "'"


source_check = query_df(
    f"SELECT COUNT(*) AS n FROM `{ANALYSIS_TABLE}`"
)
print("Connected to MatrixOne. / 已连接 MatrixOne。")
display(source_check)

In [ ]:
# Cell 4 - Fixed training and validation schedules / 固定训练与验证赛程

def build_game_frame(rows):
    """Build a consistent game schedule table from fixed rows. / 将固定赛程整理成统一表格。"""
    # Build a standard schedule table. / 生成统一格式赛程表。
    frame = pd.DataFrame(
        rows,
        columns=["season", "week", "opponent", "scheduled_kickoff_local"],
    )
    frame["scheduled_kickoff_local"] = pd.to_datetime(
        frame["scheduled_kickoff_local"]
    )
    frame["kickoff_hour"] = frame["scheduled_kickoff_local"].dt.floor("h")
    frame["game_id"] = frame.apply(
        lambda row: (
            f"{int(row['season'])}_W{int(row['week']):02d}_"
            f"{row['scheduled_kickoff_local']:%Y%m%d}"
        ),
        axis=1,
    )
    frame["game_label"] = frame.apply(
        lambda row: f"{int(row['season'])} W{int(row['week'])} vs {row['opponent']}",
        axis=1,
    )
    return frame


train_game_rows = [
    (2022, 1, "San Francisco 49ers", "2022-09-11 12:00:00"),
    (2022, 3, "Houston Texans", "2022-09-25 12:00:00"),
    (2022, 6, "Washington Commanders", "2022-10-13 19:15:00"),
    (2022, 9, "Miami Dolphins", "2022-11-06 12:00:00"),
    (2022, 10, "Detroit Lions", "2022-11-13 12:00:00"),
    (2022, 13, "Green Bay Packers", "2022-12-04 12:00:00"),
    (2022, 15, "Philadelphia Eagles", "2022-12-18 12:00:00"),
    (2022, 16, "Buffalo Bills", "2022-12-24 12:00:00"),
    (2022, 18, "Minnesota Vikings", "2023-01-08 12:00:00"),
    (2023, 1, "Green Bay Packers", "2023-09-10 15:25:00"),
    (2023, 4, "Denver Broncos", "2023-10-01 12:00:00"),
    (2023, 6, "Minnesota Vikings", "2023-10-15 12:00:00"),
    (2023, 7, "Las Vegas Raiders", "2023-10-22 12:00:00"),
    (2023, 10, "Carolina Panthers", "2023-11-09 19:15:00"),
    (2023, 14, "Detroit Lions", "2023-12-10 12:00:00"),
    (2023, 16, "Arizona Cardinals", "2023-12-24 15:25:00"),
    (2023, 17, "Atlanta Falcons", "2023-12-31 12:00:00"),
]

validation_game_rows = [
    (2024, 1, "Tennessee Titans", "2024-09-08 12:00:00"),
    (2024, 4, "Los Angeles Rams", "2024-09-29 12:00:00"),
    (2024, 5, "Carolina Panthers", "2024-10-06 12:00:00"),
    (2024, 10, "New England Patriots", "2024-11-10 12:00:00"),
    (2024, 11, "Green Bay Packers", "2024-11-17 12:00:00"),
    (2024, 12, "Minnesota Vikings", "2024-11-24 12:00:00"),
    (2024, 16, "Detroit Lions", "2024-12-22 12:00:00"),
    (2024, 17, "Seattle Seahawks", "2024-12-26 19:15:00"),
]

train_games = build_game_frame(train_game_rows)
validation_games = build_game_frame(validation_game_rows)
assert len(train_games) == 17
assert len(validation_games) == 8

print("Training games / 训练比赛:", len(train_games))
print("Validation games / 验证比赛:", len(validation_games))
display(train_games)
display(validation_games)

assumption_table = pd.DataFrame([
    {"parameter": "Platform take rate / 平台抽成", "value": PLATFORM_TAKE_RATE, "status": "scenario assumption / 情景假设"},
    {"parameter": "Variable cost per completed trip / 单笔变动成本", "value": PLATFORM_VARIABLE_COST_PER_TRIP, "status": "scenario assumption / 情景假设"},
    {"parameter": "Simulation runs / 模拟次数", "value": N_SIMULATIONS, "status": "experiment design / 实验设计"},
    {"parameter": "Training games / 训练比赛", "value": len(train_games), "status": "observed schedule / 真实赛程"},
    {"parameter": "Validation games / 验证比赛", "value": len(validation_games), "status": "observed schedule / 真实赛程"},
])
display(assumption_table)

## H3 selection and real map / H3 选择与真实地图

The public TNP locations are discrete centroid points, so geometric first-ring H3 neighbors may have no records. This notebook builds a broad eight-ring candidate area around Soldier Field, checks which candidate cells actually appear in 2022–2023, and selects the six nearest active cells within two kilometers. Geometric neighbors remain on the map only as spatial context and never receive demand or vehicles.

TNP 公开数据使用离散的区域中心点，因此几何上相邻的一环 H3 可能完全没有记录。本 Notebook 会先在 Soldier Field 周围建立八圈候选区域，再检查哪些候选格子确实出现在 2022–2023 数据中，并选择两公里内最近的六个活跃格子。几何邻居只作为地图背景，不参与需求生成，也不会获得车辆。

In [ ]:
# Cell 5 - Select observed active H3 cells around Soldier Field / 选择场馆附近有真实活动的 H3

stadium_exact_h3 = latlng_to_cell(STADIUM_LAT, STADIUM_LON, H3_RESOLUTION)
candidate_h3_cells = sorted(
    grid_disk(stadium_exact_h3, CANDIDATE_GRID_RING)
    | set(PREVALIDATED_CORE_H3)
)
candidate_sql = ", ".join(sql_string(cell) for cell in candidate_h3_cells)
observed_activity_cache = CACHE_DIR / "stadium_candidate_h3_activity_2022_2023.csv.gz"

if observed_activity_cache.exists() and not FORCE_REFRESH_DATABASE_CACHE:
    observed_h3_activity = pd.read_csv(observed_activity_cache)
    print("Loaded observed-H3 activity cache. / 已读取真实 H3 活跃度缓存。")
else:
    print("Querying broad stadium candidate activity for 2022–2023. / 正在查询2022–2023场馆周边候选H3活跃度。")
    observed_h3_activity = query_df(f"""
    SELECT
        h3,
        SUM(pickup_n) AS pickup_n,
        SUM(dropoff_n) AS dropoff_n
    FROM (
        SELECT pickup_h3 AS h3, COUNT(*) AS pickup_n, 0 AS dropoff_n
        FROM `{ANALYSIS_TABLE}`
        WHERE trip_start_timestamp >= '2022-01-01'
          AND trip_start_timestamp < '2024-01-01'
          AND pickup_h3 IS NOT NULL
          AND pickup_h3 IN ({candidate_sql})
          AND COALESCE(shared_trip_authorized, 0) = 0
        GROUP BY pickup_h3
        UNION ALL
        SELECT dropoff_h3 AS h3, 0 AS pickup_n, COUNT(*) AS dropoff_n
        FROM `{ANALYSIS_TABLE}`
        WHERE trip_end_timestamp >= '2022-01-01'
          AND trip_end_timestamp < '2024-01-01'
          AND dropoff_h3 IS NOT NULL
          AND dropoff_h3 IN ({candidate_sql})
          AND COALESCE(shared_trip_authorized, 0) = 0
        GROUP BY dropoff_h3
    ) activity
    GROUP BY h3
    """)
    observed_h3_activity.to_csv(
        observed_activity_cache, index=False, compression="gzip"
    )

observed_h3_activity["h3"] = observed_h3_activity["h3"].astype(str)
observed_h3_activity[["pickup_n", "dropoff_n"]] = (
    observed_h3_activity[["pickup_n", "dropoff_n"]].fillna(0).astype(float)
)
observed_h3_activity["total_activity"] = (
    observed_h3_activity["pickup_n"] + observed_h3_activity["dropoff_n"]
)

observed_rows = []
for row in observed_h3_activity.itertuples(index=False):
    lat, lon = cell_to_latlng(row.h3)
    observed_rows.append({
        "h3": row.h3,
        "center_lat": lat,
        "center_lon": lon,
        "distance_to_stadium_km": haversine_km(
            STADIUM_LAT, STADIUM_LON, lat, lon
        ),
        "pickup_n": row.pickup_n,
        "dropoff_n": row.dropoff_n,
        "total_activity": row.total_activity,
        "is_prevalidated_core": row.h3 in PREVALIDATED_CORE_H3,
    })
observed_metadata = pd.DataFrame(observed_rows)

eligible_active = observed_metadata[
    (observed_metadata["total_activity"] >= MIN_H3_TOTAL_ACTIVITY)
    & (observed_metadata["distance_to_stadium_km"] <= MAX_OBSERVED_H3_DISTANCE_KM)
].copy()

active_core = [
    cell for cell in PREVALIDATED_CORE_H3
    if cell in set(eligible_active["h3"])
]
if not active_core:
    raise RuntimeError(
        "No prevalidated core H3 has historical activity. / 已验证核心H3没有历史活动。"
    )

neighbor_pool = (
    eligible_active[~eligible_active["h3"].isin(active_core)]
    .sort_values(
        ["distance_to_stadium_km", "total_activity"],
        ascending=[True, False],
    )
    .head(max(N_SELECTED_ACTIVE_H3 - len(active_core), 0))
)
SELECTED_H3_CELLS = active_core + neighbor_pool["h3"].tolist()

if len(SELECTED_H3_CELLS) < 2:
    raise RuntimeError(
        "Too few active H3 cells around Soldier Field. / 场馆附近有效H3过少。"
    )

# Geometric neighbors are map context only. / 几何邻居只用于地图背景。
context_h3 = set(PREVALIDATED_CORE_H3)
for selected_cell in SELECTED_H3_CELLS:
    context_h3.update(grid_disk(selected_cell, GEOMETRIC_CONTEXT_RING))
map_h3_cells = sorted(context_h3 | set(SELECTED_H3_CELLS))

map_rows = []
activity_lookup = observed_metadata.set_index("h3")
for cell in map_h3_cells:
    lat, lon = cell_to_latlng(cell)
    if cell in activity_lookup.index:
        activity = activity_lookup.loc[cell]
        pickup_n = float(activity["pickup_n"])
        dropoff_n = float(activity["dropoff_n"])
        total_activity = float(activity["total_activity"])
    else:
        pickup_n = dropoff_n = total_activity = 0.0
    selected = cell in SELECTED_H3_CELLS
    is_core = cell in active_core
    map_rows.append({
        "h3": cell,
        "center_lat": lat,
        "center_lon": lon,
        "distance_to_stadium_km": haversine_km(
            STADIUM_LAT, STADIUM_LON, lat, lon
        ),
        "is_prevalidated_core": is_core,
        "pickup_n": pickup_n,
        "dropoff_n": dropoff_n,
        "total_activity": total_activity,
        "selected": selected,
        "selection_group": (
            "core / 核心区" if is_core else
            "active observed / 真实活跃区" if selected else
            "map context only / 仅地图背景"
        ),
        "selection_reason": (
            "Validated active stadium core / 已验证且有真实客流的场馆核心" if is_core else
            "Nearest observed active H3 within radius / 半径内最近的真实活跃H3" if selected else
            "Geometric context only; excluded from simulation / 仅几何背景，不参与模拟"
        ),
    })

h3_metadata = pd.DataFrame(map_rows).sort_values(
    ["selected", "is_prevalidated_core", "distance_to_stadium_km"],
    ascending=[False, False, True],
).reset_index(drop=True)

selected_metadata = h3_metadata[h3_metadata["selected"]].copy()
assert selected_metadata["h3"].nunique() == len(SELECTED_H3_CELLS)
assert (selected_metadata["total_activity"] >= MIN_H3_TOTAL_ACTIVITY).all()
assert set(active_core).issubset(SELECTED_H3_CELLS)

h3_metadata.to_csv(OUTPUT_DIR / "selected_h3_metadata.csv", index=False)
print("Broad geometric candidates / 大范围几何候选H3:", len(candidate_h3_cells))
print("Observed candidates in 2022–2023 / 2022–2023有数据的候选H3:", len(observed_metadata))
print("Eligible active H3 within radius / 半径内有效H3:", len(eligible_active))
print("Selected active H3 / 最终活跃H3:", len(SELECTED_H3_CELLS))
display(selected_metadata)
nearest_geometric_diagnostic = (
    h3_metadata
    .sort_values("distance_to_stadium_km")
    .head(12)[[
        "h3", "distance_to_stadium_km", "total_activity",
        "selected", "selection_group",
    ]]
)
print(
    "Nearest geometric cells may be empty because TNP publishes discrete location centroids. / "
    "几何上最近的格子可能为空，因为TNP公开的是离散地点中心点。"
)
display(nearest_geometric_diagnostic)

In [ ]:
# Cell 6 - Overlay H3 selection on a real map / 将 H3 选择叠加到真实地图

def h3_polygon(cell):
    """Convert an H3 boundary to a map polygon. / 将 H3 边界转换为地图多边形。"""
    # Convert an H3 boundary to a Shapely polygon. / 将 H3 边界转为多边形。
    boundary = cell_to_boundary(cell)
    return Polygon([(lng, lat) for lat, lng in boundary])


map_gdf = gpd.GeoDataFrame(
    h3_metadata.copy(),
    geometry=[h3_polygon(cell) for cell in h3_metadata["h3"]],
    crs="EPSG:4326",
)
map_gdf_3857 = map_gdf.to_crs(epsg=3857)
stadium_gdf = gpd.GeoDataFrame(
    {"name": [STADIUM_NAME]},
    geometry=[Point(STADIUM_LON, STADIUM_LAT)],
    crs="EPSG:4326",
).to_crs(epsg=3857)

all_bounds = map_gdf_3857.total_bounds
x_padding = (all_bounds[2] - all_bounds[0]) * 0.12
y_padding = (all_bounds[3] - all_bounds[1]) * 0.12
common_xlim = (all_bounds[0] - x_padding, all_bounds[2] + x_padding)
common_ylim = (all_bounds[1] - y_padding, all_bounds[3] + y_padding)

colors = {
    "core / 核心区": "#C62828",
    "active observed / 真实活跃区": "#F57C00",
    "map context only / 仅地图背景": "#90A4AE",
}

selected_centers_3857 = gpd.GeoDataFrame(
    selected_metadata.copy(),
    geometry=[
        Point(row.center_lon, row.center_lat)
        for row in selected_metadata.itertuples(index=False)
    ],
    crs="EPSG:4326",
).to_crs(epsg=3857)
stadium_x = float(stadium_gdf.geometry.iloc[0].x)
stadium_y = float(stadium_gdf.geometry.iloc[0].y)

fig, axes = plt.subplots(1, 2, figsize=(17, 8))
for ax, selected_only, title in [
    (axes[0], False, "All candidate H3 cells / 全部候选 H3"),
    (axes[1], True, "Nearest observed centroid cells / 最近的真实中心点格子"),
]:
    plot_frame = map_gdf_3857 if not selected_only else map_gdf_3857[map_gdf_3857["selected"]]
    for group, group_frame in plot_frame.groupby("selection_group"):
        group_frame.plot(
            ax=ax,
            facecolor=colors[group],
            edgecolor="white",
            linewidth=2,
            alpha=0.18 if group == "map context only / 仅地图背景" else 0.58,
            label=group,
        )
    stadium_gdf.plot(
        ax=ax, marker="*", color="#D50000", edgecolor="white",
        markersize=320, linewidth=1.2, label="Soldier Field",
    )
    if selected_only:
        # Distance labels explain why active centroid cells can be farther away. / 距离标注用于解释为什么有数据的中心点格子可能离场馆更远。
        for row in selected_centers_3857.itertuples(index=False):
            center_x = float(row.geometry.x)
            center_y = float(row.geometry.y)
            ax.plot(
                [stadium_x, center_x], [stadium_y, center_y],
                color="#424242", linewidth=0.8, linestyle=":", alpha=0.55,
            )
            ax.annotate(
                f"{row.distance_to_stadium_km:.2f} km",
                (center_x, center_y),
                xytext=(4, 4), textcoords="offset points",
                fontsize=7, color="#212121",
                bbox={"facecolor": "white", "alpha": 0.72, "edgecolor": "none", "pad": 1.5},
            )
    ax.set_xlim(common_xlim)
    ax.set_ylim(common_ylim)
    basemap_status = "OpenStreetMap loaded / 已加载真实地图"
    try:
        cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik, zoom=15)
    except Exception as exc:
        basemap_status = f"Basemap unavailable: {type(exc).__name__} / 地图底图未加载"
        ax.set_facecolor("#EAF3F8")
    ax.set_title(title)
    ax.set_axis_off()
    ax.legend(loc="upper right", frameon=True)

plt.suptitle(
    "Soldier Field real map and H3 selection / Soldier Field 真实地图与 H3 选择",
    fontsize=15,
)
plt.tight_layout()
static_map_path = MAP_DIR / "01_soldier_field_h3_selection_real_map.png"
plt.savefig(static_map_path, dpi=200, bbox_inches="tight")
plt.show()
print(basemap_status)
print("Static map / 静态地图:", static_map_path)

# Interactive OpenStreetMap map / 可交互 OpenStreetMap 地图
interactive_map = folium.Map(
    location=[STADIUM_LAT, STADIUM_LON],
    zoom_start=15,
    tiles="OpenStreetMap",
    control_scale=True,
)
folium.Marker(
    [STADIUM_LAT, STADIUM_LON],
    tooltip="Soldier Field",
    icon=folium.Icon(color="red", icon="star"),
).add_to(interactive_map)

for row in h3_metadata.itertuples(index=False):
    color = colors[row.selection_group]
    folium.Polygon(
        locations=cell_to_boundary(row.h3),
        color=color,
        weight=3 if row.selected else 1,
        fill=True,
        fill_color=color,
        fill_opacity=0.35 if row.selected else 0.12,
        tooltip=(
            f"H3: {row.h3}<br>"
            f"Group: {row.selection_group}<br>"
            f"Distance: {row.distance_to_stadium_km:.2f} km<br>"
            f"2022–2023 activity: {row.total_activity:,.0f}<br>"
            f"Reason: {row.selection_reason}"
        ),
    ).add_to(interactive_map)

interactive_map_path = MAP_DIR / "soldier_field_h3_selection_interactive.html"
interactive_map.save(interactive_map_path)
print("Interactive map / 交互地图:", interactive_map_path)
display(interactive_map)

## Learn demand, routes, and trip economics / 学习需求、路线和行程经济参数

MatrixOne performs the large aggregations. Python receives hourly H3 demand and compact route/fare profiles rather than all 243 million trip rows.

MatrixOne 负责大规模聚合，Python 只接收小时 H3 客流和压缩后的路线/价格分布，不会把 2.43 亿条行程全部拉入内存。

In [ ]:
# Cell 7 - Query selected-H3 hourly pickup and dropoff / 查询最终 H3 小时上下车量

selected_signature = hashlib.sha1(
    "|".join(sorted(SELECTED_H3_CELLS)).encode("utf-8")
).hexdigest()[:10]
selected_sql = ", ".join(sql_string(cell) for cell in SELECTED_H3_CELLS)
pickup_cache = CACHE_DIR / f"pickup_hourly_{selected_signature}.csv.gz"
dropoff_cache = CACHE_DIR / f"dropoff_hourly_{selected_signature}.csv.gz"

if (
    pickup_cache.exists()
    and dropoff_cache.exists()
    and not FORCE_REFRESH_DATABASE_CACHE
):
    pickup_hourly = pd.read_csv(pickup_cache, parse_dates=["hour_start"])
    dropoff_hourly = pd.read_csv(dropoff_cache, parse_dates=["hour_start"])
    print("Loaded hourly cache. / 已读取小时缓存。")
else:
    print("Query 1/2: pickup / 查询 1/2：pickup")
    pickup_hourly = query_df(f"""
    SELECT
        pickup_h3 AS h3,
        DATE_FORMAT(trip_start_timestamp, '%Y-%m-%d %H:00:00') AS hour_start,
        COUNT(*) AS pickup_count
    FROM `{ANALYSIS_TABLE}`
    WHERE pickup_h3 IN ({selected_sql})
      AND trip_start_timestamp >= '2022-01-01'
      AND trip_start_timestamp < '2025-01-01'
      AND COALESCE(shared_trip_authorized, 0) = 0
    GROUP BY pickup_h3, hour_start
    ORDER BY hour_start, h3
    """)

    print("Query 2/2: dropoff / 查询 2/2：dropoff")
    dropoff_hourly = query_df(f"""
    SELECT
        dropoff_h3 AS h3,
        DATE_FORMAT(trip_end_timestamp, '%Y-%m-%d %H:00:00') AS hour_start,
        COUNT(*) AS dropoff_count
    FROM `{ANALYSIS_TABLE}`
    WHERE dropoff_h3 IN ({selected_sql})
      AND trip_end_timestamp >= '2022-01-01'
      AND trip_end_timestamp < '2025-01-01'
      AND COALESCE(shared_trip_authorized, 0) = 0
    GROUP BY dropoff_h3, hour_start
    ORDER BY hour_start, h3
    """)
    pickup_hourly["hour_start"] = pd.to_datetime(pickup_hourly["hour_start"])
    dropoff_hourly["hour_start"] = pd.to_datetime(dropoff_hourly["hour_start"])
    pickup_hourly.to_csv(pickup_cache, index=False, compression="gzip")
    dropoff_hourly.to_csv(dropoff_cache, index=False, compression="gzip")

for frame in [pickup_hourly, dropoff_hourly]:
    frame["h3"] = frame["h3"].astype(str)
    frame["hour_start"] = pd.to_datetime(frame["hour_start"])

print("Pickup aggregate rows / pickup 聚合行数:", len(pickup_hourly))
print("Dropoff aggregate rows / dropoff 聚合行数:", len(dropoff_hourly))
display(pickup_hourly.head())
display(dropoff_hourly.head())

In [ ]:
# Cell 8 - Complete the hourly grid and learn normal baselines / 补齐小时网格并学习普通日基线

full_hours = pd.date_range(
    DATA_START, DATA_END - pd.Timedelta(hours=1), freq="h"
)
full_index = pd.MultiIndex.from_product(
    [full_hours, SELECTED_H3_CELLS], names=["hour_start", "h3"]
)
h3_hourly = full_index.to_frame(index=False)
h3_hourly = (
    h3_hourly
    .merge(pickup_hourly, on=["hour_start", "h3"], how="left")
    .merge(dropoff_hourly, on=["hour_start", "h3"], how="left")
)
h3_hourly[["pickup_count", "dropoff_count"]] = (
    h3_hourly[["pickup_count", "dropoff_count"]].fillna(0).astype(int)
)
h3_hourly["month"] = h3_hourly["hour_start"].dt.month
h3_hourly["weekday"] = h3_hourly["hour_start"].dt.weekday
h3_hourly["clock_hour"] = h3_hourly["hour_start"].dt.hour

train_event_hours = set()
for game in train_games.itertuples(index=False):
    for relative_hour in EVENT_EXCLUSION_HOURS:
        train_event_hours.add(game.kickoff_hour + pd.Timedelta(hours=relative_hour))
h3_hourly["is_train_event_window"] = h3_hourly["hour_start"].isin(train_event_hours)

eligible_train = h3_hourly[
    (h3_hourly["hour_start"] < TRAIN_END)
    & (~h3_hourly["is_train_event_window"])
].copy()

def median_absolute_deviation(series):
    """Calculate a robust spread measure around the median. / 计算围绕中位数的稳健波动范围。"""
    # Robust spread around the median. / 围绕中位数的稳健波动。
    center = series.median()
    return (series - center).abs().median()


exact_profile = (
    eligible_train
    .groupby(["h3", "month", "weekday", "clock_hour"], as_index=False)
    .agg(
        control_n=("pickup_count", "size"),
        pickup_expected_exact=("pickup_count", "median"),
        pickup_mad_exact=("pickup_count", median_absolute_deviation),
        dropoff_expected_exact=("dropoff_count", "median"),
        dropoff_mad_exact=("dropoff_count", median_absolute_deviation),
    )
)
fallback_profile = (
    eligible_train
    .groupby(["h3", "weekday", "clock_hour"], as_index=False)
    .agg(
        fallback_n=("pickup_count", "size"),
        pickup_expected_fallback=("pickup_count", "median"),
        pickup_mad_fallback=("pickup_count", median_absolute_deviation),
        dropoff_expected_fallback=("dropoff_count", "median"),
        dropoff_mad_fallback=("dropoff_count", median_absolute_deviation),
    )
)

def add_normal_baseline(frame, timestamp_col="hour_start"):
    """Add normal-hour medians and robust residuals by H3 and clock hour. / 按 H3 和小时加入普通时段中位数与稳健残差。"""
    # Attach normal values learned only from non-event training hours. / 添加仅从训练期非事件小时学习的正常值。
    out = frame.copy()
    out["month"] = out[timestamp_col].dt.month
    out["weekday"] = out[timestamp_col].dt.weekday
    out["clock_hour"] = out[timestamp_col].dt.hour
    out = out.merge(
        exact_profile, on=["h3", "month", "weekday", "clock_hour"], how="left"
    )
    out = out.merge(
        fallback_profile, on=["h3", "weekday", "clock_hour"], how="left"
    )
    use_exact = out["control_n"].fillna(0) >= MIN_NORMAL_CONTROLS
    for flow in ["pickup", "dropoff"]:
        out[f"{flow}_expected"] = np.where(
            use_exact,
            out[f"{flow}_expected_exact"],
            out[f"{flow}_expected_fallback"],
        )
        selected_mad = np.where(
            use_exact,
            out[f"{flow}_mad_exact"],
            out[f"{flow}_mad_fallback"],
        )
        poisson_floor = np.sqrt(out[f"{flow}_expected"].clip(lower=0) + 1)
        out[f"{flow}_scale"] = np.maximum.reduce([
            1.4826 * np.nan_to_num(selected_mad, nan=0.0),
            poisson_floor.to_numpy(),
            np.ones(len(out)),
        ])
    return out


h3_hourly = add_normal_baseline(h3_hourly)
h3_lookup = h3_hourly.set_index(["hour_start", "h3"])

# Use only pre-season 2024 unlabeled hours to calibrate overall market scale. / 只使用2024赛季前无标签小时校准整体客流规模。
calibration = h3_hourly[
    (h3_hourly["hour_start"] >= TRAIN_END)
    & (h3_hourly["hour_start"] < CALIBRATION_END)
].copy()
volume_scale_rows = []
for h3_cell in SELECTED_H3_CELLS:
    part = calibration[calibration["h3"] == h3_cell]
    row = {"h3": h3_cell}
    for flow in ["pickup", "dropoff"]:
        ratios = (
            part[f"{flow}_count"]
            / part[f"{flow}_expected"].replace(0, np.nan)
        ).replace([np.inf, -np.inf], np.nan).dropna()
        row[f"{flow}_volume_scale"] = float(
            np.clip(ratios.median() if len(ratios) else 1.0, 0.70, 1.50)
        )
    volume_scale_rows.append(row)
volume_scale = pd.DataFrame(volume_scale_rows).set_index("h3")

print("H3-hour rows / H3小时行数:", len(h3_hourly))
display(exact_profile.head())
display(volume_scale.reset_index())

In [ ]:
# Cell 9 - Learn event effects and realistic residual dynamics / 学习事件影响和真实连续波动

event_rows = []
for game in train_games.itertuples(index=False):
    for relative_hour in RELATIVE_HOURS:
        target_hour = game.kickoff_hour + pd.Timedelta(hours=relative_hour)
        for h3_cell in SELECTED_H3_CELLS:
            source = h3_lookup.loc[(target_hour, h3_cell)]
            event_rows.append({
                "game_id": game.game_id,
                "game_label": game.game_label,
                "relative_hour": relative_hour,
                "h3": h3_cell,
                "pickup_count": source["pickup_count"],
                "pickup_expected": source["pickup_expected"],
                "dropoff_count": source["dropoff_count"],
                "dropoff_expected": source["dropoff_expected"],
                "pickup_log_excess": np.log1p(source["pickup_count"]) - np.log1p(source["pickup_expected"]),
                "dropoff_log_excess": np.log1p(source["dropoff_count"]) - np.log1p(source["dropoff_expected"]),
            })

training_event_windows = pd.DataFrame(event_rows)

# Per-H3 templates are used by the generator. / 生成器使用逐H3事件模板。
event_template = (
    training_event_windows
    .groupby(["h3", "relative_hour"], as_index=False)
    .agg(
        pickup_log_excess=("pickup_log_excess", "median"),
        pickup_log_excess_p25=("pickup_log_excess", lambda values: values.quantile(0.25)),
        pickup_log_excess_p75=("pickup_log_excess", lambda values: values.quantile(0.75)),
        dropoff_log_excess=("dropoff_log_excess", "median"),
        dropoff_log_excess_p25=("dropoff_log_excess", lambda values: values.quantile(0.25)),
        dropoff_log_excess_p75=("dropoff_log_excess", lambda values: values.quantile(0.75)),
    )
)

# Aggregate counts first, then calculate the zone-level multiplier. / 先汇总有效H3的数量，再计算整个区域的事件倍数，避免空格子压平曲线。
zone_game_effect = (
    training_event_windows
    .groupby(["game_id", "relative_hour"], as_index=False)
    .agg(
        pickup_count=("pickup_count", "sum"),
        pickup_expected=("pickup_expected", "sum"),
        dropoff_count=("dropoff_count", "sum"),
        dropoff_expected=("dropoff_expected", "sum"),
    )
)
for flow in ["pickup", "dropoff"]:
    zone_game_effect[f"{flow}_log_excess"] = (
        np.log1p(zone_game_effect[f"{flow}_count"])
        - np.log1p(zone_game_effect[f"{flow}_expected"])
    )

zone_template = (
    zone_game_effect
    .groupby("relative_hour", as_index=False)
    .agg(
        pickup_log_excess=("pickup_log_excess", "median"),
        pickup_log_excess_p25=("pickup_log_excess", lambda values: values.quantile(0.25)),
        pickup_log_excess_p75=("pickup_log_excess", lambda values: values.quantile(0.75)),
        dropoff_log_excess=("dropoff_log_excess", "median"),
        dropoff_log_excess_p25=("dropoff_log_excess", lambda values: values.quantile(0.25)),
        dropoff_log_excess_p75=("dropoff_log_excess", lambda values: values.quantile(0.75)),
    )
)

# Event intensity distribution across the 17 training games. / 从17场训练比赛学习事件强弱分布。
game_strength = (
    training_event_windows
    .groupby("game_id", as_index=False)
    .agg(
        actual_pickup=("pickup_count", "sum"),
        expected_pickup=("pickup_expected", "sum"),
        actual_dropoff=("dropoff_count", "sum"),
        expected_dropoff=("dropoff_expected", "sum"),
    )
)
game_strength["strength_ratio"] = (
    (game_strength["actual_pickup"] + game_strength["actual_dropoff"] + 1)
    / (game_strength["expected_pickup"] + game_strength["expected_dropoff"] + 1)
)
median_strength = game_strength["strength_ratio"].median()
game_strength["relative_strength"] = (
    game_strength["strength_ratio"] / max(median_strength, 1e-9)
)
EVENT_STRENGTH_LOG_SD = float(np.clip(
    np.log(game_strength["relative_strength"].clip(lower=0.05)).std(ddof=1),
    0.08,
    0.55,
))
EVENT_STRENGTH_EMPIRICAL = game_strength["relative_strength"].to_numpy(dtype=float)
EVENT_STRENGTH_MIN = float(max(
    0.20,
    EVENT_STRENGTH_EMPIRICAL.min() * (1 - EVENT_STRENGTH_BOUND_PADDING),
))
EVENT_STRENGTH_MAX = float(min(
    2.00,
    EVENT_STRENGTH_EMPIRICAL.max() * (1 + EVENT_STRENGTH_BOUND_PADDING),
))

# Residual scale and one-hour persistence by active H3. / 按有效H3学习残差大小和相邻小时连续性。
residual_parameter_rows = []
for h3_cell in SELECTED_H3_CELLS:
    part = h3_hourly[
        (h3_hourly["h3"] == h3_cell)
        & (h3_hourly["hour_start"] < TRAIN_END)
        & (~h3_hourly["is_train_event_window"])
    ].sort_values("hour_start")
    row = {"h3": h3_cell}
    for flow in ["pickup", "dropoff"]:
        residual = (
            np.log1p(part[f"{flow}_count"].to_numpy(dtype=float))
            - np.log1p(part[f"{flow}_expected"].to_numpy(dtype=float))
        )
        low, high = np.quantile(residual, [0.02, 0.98])
        residual = np.clip(residual, low, high)
        sigma = float(np.std(residual))
        if len(residual) > 2 and sigma > 1e-8:
            rho = float(np.corrcoef(residual[:-1], residual[1:])[0, 1])
        else:
            rho = 0.0
        row[f"{flow}_residual_sigma"] = float(np.clip(sigma, 0.08, 0.80))
        row[f"{flow}_residual_rho"] = float(np.clip(np.nan_to_num(rho, nan=0.0), 0.0, 0.95))
        row[f"{flow}_residual_low"] = float(low)
        row[f"{flow}_residual_high"] = float(high)
    residual_parameter_rows.append(row)
residual_parameters = pd.DataFrame(residual_parameter_rows).set_index("h3")

event_template.to_csv(OUTPUT_DIR / "learned_event_template_by_h3.csv", index=False)
zone_template.to_csv(OUTPUT_DIR / "learned_event_template_zone.csv", index=False)
residual_parameters.reset_index().to_csv(
    OUTPUT_DIR / "residual_parameters_by_h3.csv", index=False
)

per_h3_event_diagnostics_rows = []
for h3_cell, part in event_template.groupby("h3"):
    pickup_multiplier = np.exp(part["pickup_log_excess"])
    dropoff_multiplier = np.exp(part["dropoff_log_excess"])
    per_h3_event_diagnostics_rows.append({
        "h3": h3_cell,
        "pickup_peak_multiplier": float(pickup_multiplier.max()),
        "pickup_peak_relative_hour": int(part.loc[pickup_multiplier.idxmax(), "relative_hour"]),
        "dropoff_peak_multiplier": float(dropoff_multiplier.max()),
        "dropoff_peak_relative_hour": int(part.loc[dropoff_multiplier.idxmax(), "relative_hour"]),
    })
per_h3_event_diagnostics = (
    pd.DataFrame(per_h3_event_diagnostics_rows)
    .merge(
        selected_metadata[["h3", "distance_to_stadium_km", "selection_group", "total_activity"]],
        on="h3", how="left",
    )
    .sort_values("distance_to_stadium_km")
)
per_h3_event_diagnostics.to_csv(
    OUTPUT_DIR / "selected_h3_event_sensitivity.csv", index=False
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, flow, color in [
    (axes[0], "pickup", "#1565C0"),
    (axes[1], "dropoff", "#C62828"),
]:
    median_multiplier = np.exp(zone_template[f"{flow}_log_excess"])
    p25_multiplier = np.exp(zone_template[f"{flow}_log_excess_p25"])
    p75_multiplier = np.exp(zone_template[f"{flow}_log_excess_p75"])
    ax.fill_between(
        zone_template["relative_hour"],
        p25_multiplier,
        p75_multiplier,
        color=color,
        alpha=0.18,
        label="Middle 50% of training games / 训练比赛中间50%",
    )
    ax.plot(
        zone_template["relative_hour"],
        median_multiplier,
        marker="o",
        color=color,
        linewidth=2.2,
        label="Median event multiplier / 事件倍数中位数",
    )
    ax.axhline(1, color="black", linewidth=1, label="Normal baseline / 普通日基线")
    ax.axvline(0, color="black", linestyle=":", linewidth=1.5, label="Kickoff / 开赛")
    ax.set_xticks(RELATIVE_HOURS)
    ax.set_xlabel("Hours from kickoff / 距开赛小时")
    ax.set_ylabel("Multiplier over normal / 正常值倍数")
    ax.set_title(f"{flow.title()} event effect / {flow} 事件影响")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "02_learned_event_effect.png", dpi=180, bbox_inches="tight")
plt.show()

event_diagnostics = pd.DataFrame([
    {
        "flow": flow,
        "peak_multiplier": float(np.exp(zone_template[f"{flow}_log_excess"]).max()),
        "peak_relative_hour": int(zone_template.loc[zone_template[f"{flow}_log_excess"].idxmax(), "relative_hour"]),
        "minimum_multiplier": float(np.exp(zone_template[f"{flow}_log_excess"]).min()),
    }
    for flow in ["pickup", "dropoff"]
])
assert event_diagnostics["peak_multiplier"].max() > 1.05

print("Event intensity log SD / 事件强度对数标准差:", round(EVENT_STRENGTH_LOG_SD, 4))
print(
    "Empirical event-strength range with padding / 历史事件强度加边界:",
    f"{EVENT_STRENGTH_MIN:.3f}–{EVENT_STRENGTH_MAX:.3f}",
)
display(event_diagnostics)
display(game_strength)
display(per_h3_event_diagnostics)
display(residual_parameters.reset_index())

In [ ]:
# Cell 10 - Query fare, tip, duration, and OD profiles / 查询价格、小费、时长和 OD 分布

economics_cache = CACHE_DIR / f"economics_profile_{selected_signature}.csv.gz"
route_cache = CACHE_DIR / f"route_profile_{selected_signature}.csv.gz"

if economics_cache.exists() and route_cache.exists() and not FORCE_REFRESH_DATABASE_CACHE:
    economics_profile = pd.read_csv(economics_cache)
    route_profile = pd.read_csv(route_cache)
    print("Loaded trip-profile cache. / 已读取行程分布缓存。")
else:
    print("Query 1/2: hourly economics / 查询 1/2：小时经济参数")
    economics_profile = query_df(f"""
    SELECT
        pickup_h3 AS h3,
        CAST(DATE_FORMAT(trip_start_timestamp, '%H') AS INT) AS clock_hour,
        COUNT(*) AS n,
        AVG(fare) AS avg_fare,
        AVG(fare * fare) AS mean_sq_fare,
        AVG(tip) AS avg_tip,
        AVG(CASE WHEN tip > 0 THEN 1.0 ELSE 0.0 END) AS recorded_tip_rate,
        AVG(CASE WHEN tip > 0 THEN tip ELSE NULL END) AS avg_positive_tip,
        AVG(trip_miles) AS avg_miles,
        AVG(trip_miles * trip_miles) AS mean_sq_miles,
        AVG(trip_seconds) AS avg_seconds,
        AVG(trip_seconds * trip_seconds) AS mean_sq_seconds
    FROM `{ANALYSIS_TABLE}`
    WHERE pickup_h3 IN ({selected_sql})
      AND trip_start_timestamp >= '2022-01-01'
      AND trip_start_timestamp < '2024-01-01'
      AND COALESCE(shared_trip_authorized, 0) = 0
      AND fare BETWEEN 0 AND 500
      AND tip BETWEEN 0 AND 200
      AND trip_miles BETWEEN 0 AND 100
      AND trip_seconds BETWEEN 0 AND 21600
    GROUP BY pickup_h3, clock_hour
    """)

    print("Query 2/2: OD routes / 查询 2/2：OD 路线")
    route_profile = query_df(f"""
    SELECT
        pickup_h3,
        dropoff_h3,
        COUNT(*) AS n,
        AVG(trip_miles) AS avg_miles,
        AVG(trip_miles * trip_miles) AS mean_sq_miles,
        AVG(trip_seconds) AS avg_seconds,
        AVG(trip_seconds * trip_seconds) AS mean_sq_seconds,
        AVG(fare) AS avg_fare,
        AVG(fare * fare) AS mean_sq_fare
    FROM `{ANALYSIS_TABLE}`
    WHERE pickup_h3 IN ({selected_sql})
      AND dropoff_h3 IS NOT NULL
      AND trip_start_timestamp >= '2022-01-01'
      AND trip_start_timestamp < '2024-01-01'
      AND COALESCE(shared_trip_authorized, 0) = 0
      AND fare BETWEEN 0 AND 500
      AND trip_miles BETWEEN 0 AND 100
      AND trip_seconds BETWEEN 0 AND 21600
    GROUP BY pickup_h3, dropoff_h3
    """)
    economics_profile.to_csv(economics_cache, index=False, compression="gzip")
    route_profile.to_csv(route_cache, index=False, compression="gzip")

economics_profile["h3"] = economics_profile["h3"].astype(str)
route_profile["pickup_h3"] = route_profile["pickup_h3"].astype(str)
route_profile["dropoff_h3"] = route_profile["dropoff_h3"].astype(str)

economics_fallback_rows = []
for clock_hour, group in economics_profile.groupby("clock_hour"):
    weights = group["n"].to_numpy(dtype=float)
    economics_fallback_rows.append({
        "clock_hour": int(clock_hour),
        "fallback_avg_fare": np.average(group["avg_fare"], weights=weights),
        "fallback_avg_tip": np.average(group["avg_tip"].fillna(0), weights=weights),
        "fallback_tip_rate": np.average(group["recorded_tip_rate"].fillna(0), weights=weights),
        "fallback_avg_positive_tip": np.average(group["avg_positive_tip"].fillna(0), weights=weights),
    })
economics_fallback = pd.DataFrame(economics_fallback_rows)

print("Economics profile rows / 经济参数行数:", len(economics_profile))
print("OD route rows / OD路线行数:", len(route_profile))
display(economics_profile.head())
display(route_profile.sort_values("n", ascending=False).head(15))

## Demand generator and independent validation / 需求生成器与独立验证

Each simulation receives a deterministic seed. The same generated demand world is reused across every pricing and fleet strategy, so policy comparisons are fair.

每次模拟都有固定种子。同一个随机客流世界会交给全部定价与车队策略，避免某个策略碰巧遇到低需求、另一个策略碰巧遇到高需求。

In [ ]:
# Cell 11 - Build the two-day demand generator / 构建连续两天需求生成器

event_template_lookup = event_template.set_index(["h3", "relative_hour"])

def build_simulation_timeline(kickoff):
    """Create the 48-hour event-day and control-day timeline. / 创建赛事日与对照日组成的 48 小时时间线。"""
    # Build event day + following normal day. / 构建比赛日与次日普通日。
    kickoff = pd.Timestamp(kickoff).floor("h")
    start = kickoff.normalize()
    hours = pd.date_range(start, periods=SIMULATION_HOURS, freq="h")
    timeline = pd.MultiIndex.from_product(
        [hours, SELECTED_H3_CELLS], names=["hour_start", "h3"]
    ).to_frame(index=False)
    timeline = add_normal_baseline(timeline)
    timeline["relative_hour"] = (
        (timeline["hour_start"] - kickoff) / pd.Timedelta(hours=1)
    ).astype(int)
    timeline["day_type"] = np.where(
        timeline["hour_start"].dt.normalize() == kickoff.normalize(),
        "event_day", "control_day",
    )
    for flow in ["pickup", "dropoff"]:
        timeline[f"{flow}_volume_scale"] = timeline["h3"].map(
            volume_scale[f"{flow}_volume_scale"]
        ).fillna(1.0)
        effect_map = event_template_lookup[f"{flow}_log_excess"]
        timeline[f"{flow}_event_log_effect"] = [
            float(effect_map.get((row.h3, row.relative_hour), 0.0))
            if row.day_type == "event_day" else 0.0
            for row in timeline.itertuples(index=False)
        ]
    return timeline


BASE_TIMELINE = build_simulation_timeline(SIMULATION_KICKOFF)
TIMELINE_HOURS = pd.DatetimeIndex(sorted(BASE_TIMELINE["hour_start"].unique()))
H3_ORDER = list(SELECTED_H3_CELLS)

def timeline_matrix(frame, column):
    """Align one hourly field into the simulation hour-by-H3 matrix. / 将小时字段对齐为模拟使用的小时乘 H3 矩阵。"""
    # Convert a long timeline column to hour x H3 matrix. / 将长表字段转为 小时×H3 矩阵。
    return (
        frame.pivot(index="hour_start", columns="h3", values=column)
        .reindex(index=TIMELINE_HOURS, columns=H3_ORDER)
        .to_numpy(dtype=float)
    )


BASE_INPUTS = {}
for flow in ["pickup", "dropoff"]:
    BASE_INPUTS[f"{flow}_expected"] = timeline_matrix(BASE_TIMELINE, f"{flow}_expected")
    BASE_INPUTS[f"{flow}_volume_scale"] = timeline_matrix(BASE_TIMELINE, f"{flow}_volume_scale")
    BASE_INPUTS[f"{flow}_event_log_effect"] = timeline_matrix(BASE_TIMELINE, f"{flow}_event_log_effect")

RESIDUAL_GLOBAL_SHARE = 0.35

def generate_demand_scenario(run_id, kickoff=SIMULATION_KICKOFF):
    """Generate one reproducible two-day demand world from learned distributions. / 根据已学习分布生成一个可复现的两天需求场景。"""
    # Generate one statistically plausible pickup/dropoff world. / 生成一个统计上合理的 pickup/dropoff 世界。
    seed = int(np.random.SeedSequence([RANDOM_STATE, int(run_id)]).generate_state(1)[0])
    rng = np.random.default_rng(seed)
    timeline = BASE_TIMELINE if pd.Timestamp(kickoff) == SIMULATION_KICKOFF else build_simulation_timeline(kickoff)
    hours = pd.DatetimeIndex(sorted(timeline["hour_start"].unique()))

    if pd.Timestamp(kickoff) == SIMULATION_KICKOFF:
        inputs = BASE_INPUTS
    else:
        inputs = {}
        for flow in ["pickup", "dropoff"]:
            pivot_kwargs = dict(index="hour_start", columns="h3")
            for suffix in ["expected", "volume_scale", "event_log_effect"]:
                inputs[f"{flow}_{suffix}"] = (
                    timeline.pivot(values=f"{flow}_{suffix}", **pivot_kwargs)
                    .reindex(index=hours, columns=H3_ORDER)
                    .to_numpy(dtype=float)
                )

    # Bootstrap a real game strength, add small jitter, and keep it near the / observed 17-game range. / 从真实比赛强度抽样，加入少量扰动，并限制在历史范围附近。
    empirical_strength = float(rng.choice(EVENT_STRENGTH_EMPIRICAL))
    event_strength = float(np.clip(
        empirical_strength * rng.lognormal(
            mean=-0.5 * EVENT_STRENGTH_JITTER_LOG_SD ** 2,
            sigma=EVENT_STRENGTH_JITTER_LOG_SD,
        ),
        EVENT_STRENGTH_MIN,
        EVENT_STRENGTH_MAX,
    ))
    def beta_draw(mean, concentration):
        """Draw a bounded random share around a configured mean. / 围绕设定均值生成 0 到 1 之间的随机比例。"""
        # Draw a bounded operational rate. / 抽取0到1之间的运营参数。
        alpha = mean * concentration
        beta = (1 - mean) * concentration
        return float(rng.beta(alpha, beta))

    generated = {
        "run_id": int(run_id),
        "scenario_seed": seed,
        "event_strength": event_strength,
        "dropoff_availability_rate": beta_draw(
            DROPOFF_AVAILABILITY_MEAN,
            DROPOFF_AVAILABILITY_CONCENTRATION,
        ),
        "idle_vehicle_retention_rate": beta_draw(
            IDLE_VEHICLE_RETENTION_MEAN,
            IDLE_VEHICLE_RETENTION_CONCENTRATION,
        ),
        "initial_fleet_scale": float(rng.lognormal(
            mean=-0.5 * INITIAL_FLEET_LOG_SD ** 2,
            sigma=INITIAL_FLEET_LOG_SD,
        )),
        "hours": hours,
    }

    for flow in ["pickup", "dropoff"]:
        expected = inputs[f"{flow}_expected"] * inputs[f"{flow}_volume_scale"]
        historical_event_multiplier = np.exp(
            inputs[f"{flow}_event_log_effect"]
        )
        # Scale the excess above normal linearly. Multiplying the log effect would / create unrealistic exponential tails. / 线性缩放超出正常值的部分，避免对数放大产生极端尾部。
        event_multiplier = np.clip(
            1 + event_strength * (historical_event_multiplier - 1),
            0.05,
            None,
        )
        event_effect = np.log(event_multiplier)
        shock = np.zeros_like(expected)
        global_sigma = float(
            residual_parameters[f"{flow}_residual_sigma"].median()
        )
        global_rho = float(
            residual_parameters[f"{flow}_residual_rho"].median()
        )
        global_shock = np.zeros(len(hours))
        for hour_index in range(1, len(hours)):
            global_shock[hour_index] = (
                global_rho * global_shock[hour_index - 1]
                + global_sigma * np.sqrt(max(1 - global_rho ** 2, 0))
                * rng.normal()
            )
        for cell_index, h3_cell in enumerate(H3_ORDER):
            sigma = float(residual_parameters.loc[h3_cell, f"{flow}_residual_sigma"])
            rho = float(residual_parameters.loc[h3_cell, f"{flow}_residual_rho"])
            local = np.zeros(len(hours))
            for hour_index in range(1, len(hours)):
                local[hour_index] = (
                    rho * local[hour_index - 1]
                    + sigma * np.sqrt(max(1 - rho ** 2, 0)) * rng.normal()
                )
            combined_shock = (
                RESIDUAL_GLOBAL_SHARE * global_shock
                + np.sqrt(1 - RESIDUAL_GLOBAL_SHARE ** 2) * local
            )
            shock[:, cell_index] = np.clip(
                combined_shock,
                residual_parameters.loc[h3_cell, f"{flow}_residual_low"],
                residual_parameters.loc[h3_cell, f"{flow}_residual_high"],
            )
        log_mean = np.log1p(np.clip(expected, 0, None)) + event_effect + shock
        intensity = np.clip(np.expm1(log_mean), 0, None)
        generated[flow] = rng.poisson(intensity).astype(np.int32)
        generated[f"{flow}_expected"] = expected
    return generated


example_demand = generate_demand_scenario(0)
print("Scenario seed / 场景种子:", example_demand["scenario_seed"])
print("Event strength / 事件强度:", round(example_demand["event_strength"], 3))
print("Pickup requests / pickup请求:", f"{example_demand['pickup'].sum():,}")
print("Dropoff arrivals / dropoff到达:", f"{example_demand['dropoff'].sum():,}")
print("Dropoff availability / dropoff转可用车辆比例:", round(example_demand["dropoff_availability_rate"], 3))
print("Idle retention / 空闲车辆小时留存率:", round(example_demand["idle_vehicle_retention_rate"], 3))
print("Initial fleet scale / 初始车队随机缩放:", round(example_demand["initial_fleet_scale"], 3))

In [ ]:
# Cell 12 - Validate the demand generator on unseen 2024 games / 在未见过的2024比赛上验证需求生成器

N_DEMAND_VALIDATION_DRAWS = 300
validation_records = []

for game_index, game in enumerate(tqdm(validation_games.itertuples(index=False), total=len(validation_games))):
    actual_hours = [
        game.kickoff_hour + pd.Timedelta(hours=relative_hour)
        for relative_hour in RELATIVE_HOURS
    ]
    actual = (
        h3_hourly[h3_hourly["hour_start"].isin(actual_hours)]
        .groupby("hour_start")[["pickup_count", "dropoff_count"]]
        .sum()
        .reindex(actual_hours)
    )
    draws = {"pickup": [], "dropoff": []}
    for draw_index in range(N_DEMAND_VALIDATION_DRAWS):
        generated = generate_demand_scenario(
            run_id=1_000_000 + game_index * 10_000 + draw_index,
            kickoff=game.kickoff_hour,
        )
        generated_hours = generated["hours"]
        hour_positions = [int(np.where(generated_hours == hour)[0][0]) for hour in actual_hours]
        for flow in ["pickup", "dropoff"]:
            draws[flow].append(generated[flow][hour_positions].sum(axis=1))
    for flow in ["pickup", "dropoff"]:
        draw_array = np.asarray(draws[flow])
        p05, p50, p95 = np.quantile(draw_array, [0.05, 0.50, 0.95], axis=0)
        actual_values = actual[f"{flow}_count"].to_numpy(dtype=float)
        for index, relative_hour in enumerate(RELATIVE_HOURS):
            validation_records.append({
                "game_id": game.game_id,
                "game_label": game.game_label,
                "flow": flow,
                "relative_hour": relative_hour,
                "actual": actual_values[index],
                "p05": p05[index],
                "p50": p50[index],
                "p95": p95[index],
            })

demand_validation = pd.DataFrame(validation_records)
demand_validation["inside_90"] = (
    (demand_validation["actual"] >= demand_validation["p05"])
    & (demand_validation["actual"] <= demand_validation["p95"])
)
demand_validation["absolute_error"] = (
    demand_validation["actual"] - demand_validation["p50"]
).abs()
validation_metrics = (
    demand_validation
    .groupby("flow", as_index=False)
    .agg(
        MAE=("absolute_error", "mean"),
        interval_90_coverage=("inside_90", "mean"),
        actual_total=("actual", "sum"),
        predicted_total=("p50", "sum"),
    )
)
validation_metrics["WAPE"] = (
    demand_validation.groupby("flow")["absolute_error"].sum().to_numpy()
    / validation_metrics["actual_total"].replace(0, np.nan)
)
demand_validation.to_csv(OUTPUT_DIR / "demand_generator_2024_validation.csv", index=False)
validation_metrics.to_csv(OUTPUT_DIR / "demand_generator_2024_metrics.csv", index=False)
display(validation_metrics)

average_validation = (
    demand_validation
    .groupby(["flow", "relative_hour"], as_index=False)[["actual", "p05", "p50", "p95"]]
    .mean()
)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, flow, color in [(axes[0], "pickup", "#1565C0"), (axes[1], "dropoff", "#C62828")]:
    part = average_validation[average_validation["flow"] == flow]
    ax.fill_between(part["relative_hour"], part["p05"], part["p95"], color=color, alpha=0.18, label="Generated 90% interval")
    ax.plot(part["relative_hour"], part["p50"], color="#2E7D32", linestyle="--", linewidth=2.2, label="Generated P50")
    ax.plot(part["relative_hour"], part["actual"], color=color, marker="o", linewidth=2.2, label="2024 actual")
    ax.axvline(0, color="black", linestyle=":")
    ax.set_title(f"2024 {flow} validation / 2024 {flow} 验证")
    ax.set_xlabel("Hours from kickoff / 距开赛小时")
    ax.set_ylabel("Trips / 完成订单")
    ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03_demand_generator_2024_validation.png", dpi=180, bbox_inches="tight")
plt.show()

## Fleet, pricing, and contribution-profit simulation / 车辆、定价与贡献利润模拟

The vehicle stock is an explicit local-supply proxy rather than an observed driver count. Only a sampled share of generated dropoffs becomes locally available, and the fleet scenario scales that share throughout the full two-day period instead of changing only the opening inventory. Idle vehicles may leave the small analysis zone each hour. These assumptions prevent vehicles from accumulating forever and keep fleet stress visible after the first few hours. Every policy is evaluated on the same demand and supply world.

车辆库存是明确的本地供给代理，不是真实司机数量。每次生成的 dropoff 只有随机抽取的一部分会成为本地可用车辆，车队情景会在完整两天中持续缩放这一比例，而不是只改变开场车辆数。空闲车辆每小时也可能离开这个小区域，从而避免车辆无限累积，并让车队压力在前几个小时之后仍然有效。每种策略都会在完全相同的需求与供给世界中接受测试。

Dynamic pricing can reduce retained requests and attract additional supply. The output reports both operational service rate and fulfillment of original base demand, so demand suppressed by price is not mistaken for improved service.

动态定价可能减少愿意继续叫车的请求，也可能吸引额外供给。结果会同时报告运营完成率和原始需求满足率，避免把“被价格压掉的需求”误认为服务质量提升。

In [ ]:
# Cell 13 - Prepare fleet sizes, economics, and strategy configurations / 准备车辆、经济参数与策略组合

normal_pickup_p95 = (
    eligible_train
    .groupby("h3")["pickup_count"]
    .quantile(0.95)
    .reindex(H3_ORDER)
)
if normal_pickup_p95.isna().any() or (normal_pickup_p95 <= 0).any():
    raise RuntimeError(
        "Selected H3 contains no usable pickup history. / 最终H3包含无有效pickup历史的格子。"
    )

# Reference capacity equals the historical non-event hourly pickup P95. / 参考车辆容量等于历史非事件小时pickup的P95。
REFERENCE_INITIAL_FLEET = np.maximum(
    np.ceil(normal_pickup_p95.to_numpy()), 5
).astype(int)

economics_lookup = economics_profile.set_index(["h3", "clock_hour"])
fallback_lookup = economics_fallback.set_index("clock_hour")

def build_economics_matrices(hours):
    """Build fare, tip, distance, and duration inputs by hour and H3. / 构建按小时和 H3 的车费、小费、距离与时长输入。"""
    # Build expected fare and tip matrices by hour and H3. / 构建按小时和H3的预期价格与小费矩阵。
    fare = np.zeros((len(hours), len(H3_ORDER)), dtype=float)
    tip = np.zeros_like(fare)
    tip_rate = np.zeros_like(fare)
    positive_tip = np.zeros_like(fare)
    for hour_index, timestamp in enumerate(hours):
        for cell_index, h3_cell in enumerate(H3_ORDER):
            key = (h3_cell, timestamp.hour)
            if key in economics_lookup.index:
                row = economics_lookup.loc[key]
                fare[hour_index, cell_index] = float(row["avg_fare"])
                tip[hour_index, cell_index] = float(np.nan_to_num(row["avg_tip"], nan=0.0))
                tip_rate[hour_index, cell_index] = float(np.nan_to_num(row["recorded_tip_rate"], nan=0.0))
                positive_tip[hour_index, cell_index] = float(np.nan_to_num(row["avg_positive_tip"], nan=0.0))
            else:
                fallback = fallback_lookup.loc[timestamp.hour]
                fare[hour_index, cell_index] = float(fallback["fallback_avg_fare"])
                tip[hour_index, cell_index] = float(fallback["fallback_avg_tip"])
                tip_rate[hour_index, cell_index] = float(fallback["fallback_tip_rate"])
                positive_tip[hour_index, cell_index] = float(fallback["fallback_avg_positive_tip"])
    return {
        "avg_fare": np.clip(fare, 1.0, 200.0),
        "avg_tip": np.clip(tip, 0.0, 100.0),
        "tip_rate": np.clip(tip_rate, 0.0, 1.0),
        "avg_positive_tip": np.clip(positive_tip, 0.0, 100.0),
    }


ECONOMICS_MATRICES = build_economics_matrices(TIMELINE_HOURS)
CONFIGURATIONS = [
    {
        "fleet_name": fleet_name,
        "fleet_factor": fleet_factor,
        "strategy_name": strategy_name,
        **strategy,
    }
    for fleet_name, fleet_factor in FLEET_SCENARIOS.items()
    for strategy_name, strategy in PRICING_STRATEGIES.items()
]
STANDARD_HOURLY_CONFIGS = [
    config for config in CONFIGURATIONS if config["fleet_name"] == "standard"
]

fleet_table = pd.DataFrame({
    "h3": H3_ORDER,
    "normal_pickup_p95": normal_pickup_p95.to_numpy(),
    "reference_initial_vehicles": REFERENCE_INITIAL_FLEET,
})
for fleet_name, fleet_factor in FLEET_SCENARIOS.items():
    fleet_table[f"{fleet_name}_initial_vehicles"] = np.maximum(
        np.round(REFERENCE_INITIAL_FLEET * fleet_factor), 1
    ).astype(int)

display(fleet_table)
display(pd.DataFrame(CONFIGURATIONS))

In [ ]:
# Cell 14 - Operating-policy simulator / 经营策略模拟器

def move_surplus_to_shortage(available, requested):
    """Move a limited share of nearby surplus vehicles toward shortages. / 将有限比例的周边富余车辆调向缺车区域。"""
    # Move a limited share of surplus vehicles within active selected cells. / 只在有真实活动的最终格子之间有限调配富余车辆。
    available = available.copy()
    moved = 0
    shortage_cells = np.where(requested > available)[0]
    for target in shortage_cells:
        remaining_shortage = int(max(requested[target] - available[target], 0))
        if remaining_shortage <= 0:
            continue
        surplus = np.maximum(available - MIN_LOCAL_RESERVE, 0)
        surplus[target] = 0
        transferable = np.floor(surplus * LOCAL_REBALANCE_SHARE).astype(int)
        for source in np.argsort(-transferable):
            amount = int(min(remaining_shortage, transferable[source]))
            if amount <= 0:
                continue
            available[source] -= amount
            available[target] += amount
            moved += amount
            remaining_shortage -= amount
            if remaining_shortage <= 0:
                break
    return available, moved


def run_operating_policy(demand, config, return_cell_detail=False):
    """Apply one fleet and pricing policy to a generated demand world. / 在一个生成场景中执行一组车队与定价策略。"""
    # Apply one fleet and pricing policy to one shared demand-and-supply world. / 将一个车队与定价策略应用到同一个需求和供给世界。
    base_requests = demand["pickup"].astype(int)
    base_dropoffs = demand["dropoff"].astype(int)
    hours = demand["hours"]
    dropoff_availability_rate = float(demand["dropoff_availability_rate"])
    idle_vehicle_retention_rate = float(demand["idle_vehicle_retention_rate"])
    initial_fleet_scale = float(demand["initial_fleet_scale"])
    effective_dropoff_availability_rate = float(np.clip(
        dropoff_availability_rate * config["fleet_factor"], 0.0, 1.0
    ))

    initial_fleet = np.maximum(
        np.round(
            REFERENCE_INITIAL_FLEET
            * config["fleet_factor"]
            * initial_fleet_scale
        ),
        1,
    ).astype(int)
    available = initial_fleet.copy()
    shape = base_requests.shape

    retained_vehicle_matrix = np.zeros(shape, dtype=np.int32)
    effective_matrix = np.zeros(shape, dtype=np.int32)
    suppressed_matrix = np.zeros(shape, dtype=np.int32)
    inbound_matrix = np.zeros(shape, dtype=np.int32)
    extra_supply_matrix = np.zeros(shape, dtype=np.int32)
    completed_matrix = np.zeros(shape, dtype=np.int32)
    unserved_matrix = np.zeros(shape, dtype=np.int32)
    available_end_matrix = np.zeros(shape, dtype=np.int32)
    price_matrix = np.ones(shape, dtype=np.float32)
    rider_spend_matrix = np.zeros(shape, dtype=np.float64)
    revenue_matrix = np.zeros(shape, dtype=np.float64)
    incentive_matrix = np.zeros(shape, dtype=np.float64)
    cost_matrix = np.zeros(shape, dtype=np.float64)
    profit_matrix = np.zeros(shape, dtype=np.float64)
    utilization_matrix = np.zeros(shape, dtype=np.float32)
    rebalanced_by_hour = np.zeros(len(hours), dtype=np.int32)

    for hour_index, timestamp in enumerate(hours):
        # Idle vehicles can leave the small zone instead of accumulating forever. / 空闲车辆可能离开小区域，避免车辆无限累积。
        retained_vehicles = np.floor(
            available * idle_vehicle_retention_rate
        ).astype(int)
        inbound = np.floor(
            base_dropoffs[hour_index] * effective_dropoff_availability_rate
        ).astype(int)
        available_before = retained_vehicles + inbound

        pressure = base_requests[hour_index] / np.maximum(available_before, 1)
        price_multiplier = np.clip(
            1 + config["price_pressure_coefficient"] * np.maximum(pressure - 0.80, 0),
            1.0,
            config["max_price_multiplier"],
        )
        demand_retention = np.clip(
            np.power(price_multiplier, config["demand_elasticity"]),
            0.0,
            1.0,
        )
        effective_requests = np.floor(
            base_requests[hour_index] * demand_retention
        ).astype(int)
        suppressed_requests = base_requests[hour_index] - effective_requests

        # Surge may attract additional nearby supply. / 动态价格可能吸引额外附近供给。
        extra_supply = np.floor(
            inbound * config["supply_response"] * (price_multiplier - 1)
        ).astype(int)
        available_before += extra_supply

        available_after_move, moved = move_surplus_to_shortage(
            available_before, effective_requests
        )
        completed = np.minimum(effective_requests, available_after_move)
        unserved = effective_requests - completed
        available = available_after_move - completed

        average_fare = ECONOMICS_MATRICES["avg_fare"][hour_index]
        average_tip = ECONOMICS_MATRICES["avg_tip"][hour_index]
        fare_booking = completed * average_fare * price_multiplier
        rider_spend = fare_booking + completed * average_tip
        platform_revenue = fare_booking * PLATFORM_TAKE_RATE
        incremental_surge = completed * average_fare * (price_multiplier - 1)
        incentive_cost = incremental_surge * config["driver_incentive_share"]
        variable_cost = completed * PLATFORM_VARIABLE_COST_PER_TRIP
        contribution_profit = platform_revenue - incentive_cost - variable_cost
        utilization = completed / np.maximum(available_after_move, 1)

        retained_vehicle_matrix[hour_index] = retained_vehicles
        effective_matrix[hour_index] = effective_requests
        suppressed_matrix[hour_index] = suppressed_requests
        inbound_matrix[hour_index] = inbound
        extra_supply_matrix[hour_index] = extra_supply
        completed_matrix[hour_index] = completed
        unserved_matrix[hour_index] = unserved
        available_end_matrix[hour_index] = available
        price_matrix[hour_index] = price_multiplier
        rider_spend_matrix[hour_index] = rider_spend
        revenue_matrix[hour_index] = platform_revenue
        incentive_matrix[hour_index] = incentive_cost
        cost_matrix[hour_index] = variable_cost
        profit_matrix[hour_index] = contribution_profit
        utilization_matrix[hour_index] = utilization
        rebalanced_by_hour[hour_index] = moved

    event_hour_mask = np.asarray([
        timestamp.normalize() == SIMULATION_KICKOFF.normalize()
        for timestamp in hours
    ])
    summaries = []
    for period, hour_mask in [
        ("event_day", event_hour_mask),
        ("control_day", ~event_hour_mask),
        ("overall", np.ones(len(hours), dtype=bool)),
    ]:
        total_base_requests = int(base_requests[hour_mask].sum())
        total_effective_requests = int(effective_matrix[hour_mask].sum())
        total_suppressed = int(suppressed_matrix[hour_mask].sum())
        total_completed = int(completed_matrix[hour_mask].sum())
        total_unserved = int(unserved_matrix[hour_mask].sum())
        total_lost = total_suppressed + total_unserved
        price_values = price_matrix[hour_mask].ravel()
        price_weights = np.maximum(completed_matrix[hour_mask].ravel(), 1)
        summaries.append({
            "run_id": demand["run_id"],
            "scenario_seed": demand["scenario_seed"],
            "event_strength": demand["event_strength"],
            "dropoff_availability_rate": dropoff_availability_rate,
            "effective_dropoff_availability_rate": effective_dropoff_availability_rate,
            "idle_vehicle_retention_rate": idle_vehicle_retention_rate,
            "initial_fleet_scale": initial_fleet_scale,
            "initial_vehicles": int(initial_fleet.sum()),
            "fleet_name": config["fleet_name"],
            "strategy_name": config["strategy_name"],
            "period": period,
            "base_requests": total_base_requests,
            "total_requests": total_effective_requests,
            "price_suppressed_requests": total_suppressed,
            "completed_trips": total_completed,
            "unserved_trips": total_unserved,
            "lost_demand": total_lost,
            "service_rate": total_completed / max(total_effective_requests, 1),
            "base_demand_fulfillment_rate": total_completed / max(total_base_requests, 1),
            "unserved_rate": total_unserved / max(total_base_requests, 1),
            "lost_demand_rate": total_lost / max(total_base_requests, 1),
            "stockout_flag": int(total_unserved > 0),
            "severe_shortage_flag": int(total_unserved / max(total_base_requests, 1) >= 0.05),
            "rider_spend": float(rider_spend_matrix[hour_mask].sum()),
            "platform_revenue": float(revenue_matrix[hour_mask].sum()),
            "driver_incentive_cost": float(incentive_matrix[hour_mask].sum()),
            "variable_operating_cost": float(cost_matrix[hour_mask].sum()),
            "contribution_profit": float(profit_matrix[hour_mask].sum()),
            "max_hourly_utilization": float(utilization_matrix[hour_mask].mean(axis=1).max()),
            "mean_price_multiplier": float(np.average(price_values, weights=price_weights)),
            "max_price_multiplier": float(price_values.max()),
        })

    cell_detail = None
    if return_cell_detail:
        rows = []
        for hour_index, timestamp in enumerate(hours):
            day_type = "event_day" if event_hour_mask[hour_index] else "control_day"
            for cell_index, h3_cell in enumerate(H3_ORDER):
                rows.append({
                    "hour_start": timestamp,
                    "day_type": day_type,
                    "h3": h3_cell,
                    "base_requests": int(base_requests[hour_index, cell_index]),
                    "effective_requests": int(effective_matrix[hour_index, cell_index]),
                    "price_suppressed_requests": int(suppressed_matrix[hour_index, cell_index]),
                    "base_dropoff_arrivals": int(base_dropoffs[hour_index, cell_index]),
                    "dropoff_arrivals": int(inbound_matrix[hour_index, cell_index]),
                    "retained_vehicles": int(retained_vehicle_matrix[hour_index, cell_index]),
                    "extra_supply": int(extra_supply_matrix[hour_index, cell_index]),
                    "completed": int(completed_matrix[hour_index, cell_index]),
                    "unserved": int(unserved_matrix[hour_index, cell_index]),
                    "available_end": int(available_end_matrix[hour_index, cell_index]),
                    "price_multiplier": float(price_matrix[hour_index, cell_index]),
                    "rider_spend": float(rider_spend_matrix[hour_index, cell_index]),
                    "platform_revenue": float(revenue_matrix[hour_index, cell_index]),
                    "incentive_cost": float(incentive_matrix[hour_index, cell_index]),
                    "variable_cost": float(cost_matrix[hour_index, cell_index]),
                    "contribution_profit": float(profit_matrix[hour_index, cell_index]),
                    "utilization": float(utilization_matrix[hour_index, cell_index]),
                    "rebalanced_zone_total": int(rebalanced_by_hour[hour_index]),
                })
        cell_detail = pd.DataFrame(rows)
    return pd.DataFrame(summaries), cell_detail


dry_demand = generate_demand_scenario(1)
dry_summary, dry_detail = run_operating_policy(
    dry_demand,
    next(
        config for config in CONFIGURATIONS
        if config["fleet_name"] == "standard"
        and config["strategy_name"] == "balanced_surge"
    ),
    return_cell_detail=True,
)
display(dry_summary)
display(dry_detail.head())

In [ ]:
# Cell 15 - Validate a small batch before the 20,000-run experiment / 全量运行前先验证小批次

SMOKE_TEST_RUNS = 50
smoke_rows = []
for run_id in range(SMOKE_TEST_RUNS):
    demand = generate_demand_scenario(run_id)
    for config in CONFIGURATIONS:
        summary, _ = run_operating_policy(demand, config)
        smoke_rows.append(summary)
smoke_summary = pd.concat(smoke_rows, ignore_index=True)

smoke_checks = pd.Series({
    "expected_rows": len(smoke_summary) == SMOKE_TEST_RUNS * len(CONFIGURATIONS) * 3,
    "no_negative_base_requests": bool((smoke_summary["base_requests"] >= 0).all()),
    "effective_not_above_base": bool((smoke_summary["total_requests"] <= smoke_summary["base_requests"]).all()),
    "completed_not_above_effective": bool((smoke_summary["completed_trips"] <= smoke_summary["total_requests"]).all()),
    "suppressed_plus_effective_equals_base": bool(
        (smoke_summary["price_suppressed_requests"] + smoke_summary["total_requests"] == smoke_summary["base_requests"]).all()
    ),
    "service_rate_in_range": bool(smoke_summary["service_rate"].between(0, 1).all()),
    "base_fulfillment_in_range": bool(smoke_summary["base_demand_fulfillment_rate"].between(0, 1).all()),
    "supply_rates_in_range": bool(
        smoke_summary["dropoff_availability_rate"].between(0, 1).all()
        and smoke_summary["idle_vehicle_retention_rate"].between(0, 1).all()
    ),
    "finite_profit": bool(np.isfinite(smoke_summary["contribution_profit"]).all()),
})
display(smoke_checks.to_frame("passed"))
if not smoke_checks.all():
    raise RuntimeError("Smoke test failed. / 小批次检查失败。")

smoke_diagnostic = (
    smoke_summary[smoke_summary["period"] == "overall"]
    .groupby(["fleet_name", "strategy_name"], as_index=False)
    .agg(
        mean_base_fulfillment=("base_demand_fulfillment_rate", "mean"),
        any_stockout_rate=("stockout_flag", "mean"),
        severe_shortage_rate=("severe_shortage_flag", "mean"),
        mean_price_multiplier=("mean_price_multiplier", "mean"),
        mean_profit=("contribution_profit", "mean"),
    )
)
display(smoke_diagnostic)
print("Smoke test passed. / 小批次检查通过。")

In [ ]:
# Cell 16 - Run 20,000 Monte Carlo scenarios with checkpoints / 分批运行20,000次蒙特卡洛模拟

simulation_config_for_hash = {
    "simulator_version": SIMULATOR_VERSION,
    "selected_h3": sorted(SELECTED_H3_CELLS),
    "h3_selection": {
        "count": N_SELECTED_ACTIVE_H3,
        "maximum_distance_km": MAX_OBSERVED_H3_DISTANCE_KM,
        "minimum_total_activity": MIN_H3_TOTAL_ACTIVITY,
        "candidate_grid_ring": CANDIDATE_GRID_RING,
    },
    "kickoff": str(SIMULATION_KICKOFF),
    "runs": N_SIMULATIONS,
    "batch_size": BATCH_SIZE,
    "random_state": RANDOM_STATE,
    "fleet_scenarios": FLEET_SCENARIOS,
    "pricing_strategies": PRICING_STRATEGIES,
    "dropoff_availability_mean": DROPOFF_AVAILABILITY_MEAN,
    "dropoff_availability_concentration": DROPOFF_AVAILABILITY_CONCENTRATION,
    "idle_vehicle_retention_mean": IDLE_VEHICLE_RETENTION_MEAN,
    "idle_vehicle_retention_concentration": IDLE_VEHICLE_RETENTION_CONCENTRATION,
    "initial_fleet_log_sd": INITIAL_FLEET_LOG_SD,
    "event_strength_sampling": "empirical_bootstrap_with_bounded_jitter",
    "event_strength_min": EVENT_STRENGTH_MIN,
    "event_strength_max": EVENT_STRENGTH_MAX,
    "event_strength_jitter_log_sd": EVENT_STRENGTH_JITTER_LOG_SD,
    "residual_shock_bounds": "historical_2nd_to_98th_percentile_by_h3",
    "local_rebalance_share": LOCAL_REBALANCE_SHARE,
    "minimum_local_reserve": MIN_LOCAL_RESERVE,
    "take_rate": PLATFORM_TAKE_RATE,
    "variable_cost": PLATFORM_VARIABLE_COST_PER_TRIP,
}
CONFIG_SIGNATURE = hashlib.sha1(
    json.dumps(simulation_config_for_hash, sort_keys=True).encode("utf-8")
).hexdigest()[:12]
MONTE_CARLO_DIR = MONTE_CARLO_ROOT / CONFIG_SIGNATURE
MONTE_CARLO_DIR.mkdir(parents=True, exist_ok=True)
(MONTE_CARLO_DIR / "configuration.json").write_text(
    json.dumps(simulation_config_for_hash, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

def run_simulation_batch(start_run, end_run):
    """Run and save one checkpoint batch of Monte Carlo worlds. / 运行并保存一批蒙特卡洛场景检查点。"""
    # Run a checkpointable batch. / 运行一个可断点续跑的批次。
    summary_frames = []
    hourly_records = []
    standard_keys = {
        (config["fleet_name"], config["strategy_name"])
        for config in STANDARD_HOURLY_CONFIGS
    }
    for run_id in range(start_run, end_run):
        demand = generate_demand_scenario(run_id)
        for config in CONFIGURATIONS:
            keep_hourly = (
                config["fleet_name"], config["strategy_name"]
            ) in standard_keys
            summary, detail = run_operating_policy(
                demand, config, return_cell_detail=keep_hourly
            )
            summary_frames.append(summary)
            if keep_hourly:
                zone_hourly_detail = (
                    detail.groupby(["hour_start", "day_type"], as_index=False)
                    .agg(
                        base_requests=("base_requests", "sum"),
                        effective_requests=("effective_requests", "sum"),
                        price_suppressed_requests=("price_suppressed_requests", "sum"),
                        base_dropoff_arrivals=("base_dropoff_arrivals", "sum"),
                        available_inbound=("dropoff_arrivals", "sum"),
                        retained_vehicles=("retained_vehicles", "sum"),
                        extra_supply=("extra_supply", "sum"),
                        completed=("completed", "sum"),
                        unserved=("unserved", "sum"),
                        available_end=("available_end", "sum"),
                        rider_spend=("rider_spend", "sum"),
                        contribution_profit=("contribution_profit", "sum"),
                        mean_price_multiplier=("price_multiplier", "mean"),
                    )
                )
                zone_hourly_detail["run_id"] = run_id
                zone_hourly_detail["strategy_name"] = config["strategy_name"]
                hourly_records.append(zone_hourly_detail)
    return (
        pd.concat(summary_frames, ignore_index=True),
        pd.concat(hourly_records, ignore_index=True),
    )


if RUN_FULL_MONTE_CARLO:
    batch_starts = list(range(0, N_SIMULATIONS, BATCH_SIZE))
    for start_run in tqdm(batch_starts, desc="Monte Carlo batches / 蒙特卡洛批次"):
        end_run = min(start_run + BATCH_SIZE, N_SIMULATIONS)
        summary_path = MONTE_CARLO_DIR / f"summary_{start_run:05d}_{end_run:05d}.csv.gz"
        hourly_path = MONTE_CARLO_DIR / f"hourly_{start_run:05d}_{end_run:05d}.csv.gz"
        if (
            summary_path.exists()
            and hourly_path.exists()
            and not FORCE_RERUN_MONTE_CARLO
        ):
            continue
        batch_summary, batch_hourly = run_simulation_batch(start_run, end_run)
        batch_summary.to_csv(summary_path, index=False, compression="gzip")
        batch_hourly.to_csv(hourly_path, index=False, compression="gzip")
        print(f"Saved runs {start_run:,}–{end_run - 1:,} / 已保存模拟 {start_run:,}–{end_run - 1:,}")
else:
    print("RUN_FULL_MONTE_CARLO=False; skipped full simulation. / 已跳过全量模拟。")

print("Simulator version / 模拟器版本:", SIMULATOR_VERSION)
print("Configuration signature / 配置签名:", CONFIG_SIGNATURE)
print("Checkpoint directory / 断点目录:", MONTE_CARLO_DIR)

In [ ]:
# Cell 17 - Load and verify all Monte Carlo outputs / 读取并验证全部模拟结果

summary_files = sorted(MONTE_CARLO_DIR.glob("summary_*.csv.gz"))
hourly_files = sorted(MONTE_CARLO_DIR.glob("hourly_*.csv.gz"))
if not summary_files or not hourly_files:
    raise FileNotFoundError(
        "Monte Carlo outputs are missing. Run Cell 16 first. / 缺少模拟结果，请先运行 Cell 16。"
    )

simulation_summary = pd.concat(
    [pd.read_csv(path) for path in summary_files], ignore_index=True
)
simulation_hourly = pd.concat(
    [pd.read_csv(path, parse_dates=["hour_start"]) for path in hourly_files],
    ignore_index=True,
)

expected_policy_count = len(CONFIGURATIONS)
expected_summary_rows = N_SIMULATIONS * expected_policy_count * 3
load_checks = pd.Series({
    "all_run_ids_present": simulation_summary["run_id"].nunique() == N_SIMULATIONS,
    "summary_row_count": len(simulation_summary) == expected_summary_rows,
    "all_policy_combinations": (
        simulation_summary[["fleet_name", "strategy_name"]]
        .drop_duplicates().shape[0] == expected_policy_count
    ),
    "three_periods": set(simulation_summary["period"]) == {"event_day", "control_day", "overall"},
    "effective_not_above_base": bool((simulation_summary["total_requests"] <= simulation_summary["base_requests"]).all()),
    "completed_not_above_effective": bool((simulation_summary["completed_trips"] <= simulation_summary["total_requests"]).all()),
    "suppressed_plus_effective_equals_base": bool(
        (
            simulation_summary["price_suppressed_requests"]
            + simulation_summary["total_requests"]
            == simulation_summary["base_requests"]
        ).all()
    ),
    "service_rate_in_range": bool(simulation_summary["service_rate"].between(0, 1).all()),
    "base_fulfillment_in_range": bool(simulation_summary["base_demand_fulfillment_rate"].between(0, 1).all()),
    "no_negative_lost_demand": bool((simulation_summary["lost_demand"] >= 0).all()),
    "supply_rates_in_range": bool(
        simulation_summary["dropoff_availability_rate"].between(0, 1).all()
        and simulation_summary["idle_vehicle_retention_rate"].between(0, 1).all()
    ),
})
display(load_checks.to_frame("passed"))
if not load_checks.all():
    raise RuntimeError("Monte Carlo output validation failed. / 蒙特卡洛结果检查失败。")

simulation_summary.to_csv(
    OUTPUT_DIR / "monte_carlo_20000_business_summary.csv.gz",
    index=False,
    compression="gzip",
)
print("Policy combinations / 策略组合:", expected_policy_count)
print("Summary rows / 汇总行数:", f"{len(simulation_summary):,}")
print("Hourly rows / 小时行数:", f"{len(simulation_hourly):,}")

In [ ]:
# Cell 18 - Generated event-day and control-day demand intervals / 从20,000次生成数据统计比赛日与普通日区间

standard_fixed_hourly = simulation_hourly[
    simulation_hourly["strategy_name"] == "fixed_price"
].copy()
demand_interval = (
    standard_fixed_hourly
    .groupby("hour_start")
    .agg(
        pickup_p05=("base_requests", lambda values: values.quantile(0.05)),
        pickup_p50=("base_requests", "median"),
        pickup_p95=("base_requests", lambda values: values.quantile(0.95)),
        dropoff_p05=("base_dropoff_arrivals", lambda values: values.quantile(0.05)),
        dropoff_p50=("base_dropoff_arrivals", "median"),
        dropoff_p95=("base_dropoff_arrivals", lambda values: values.quantile(0.95)),
    )
    .reset_index()
)

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)
for ax, flow, color in [
    (axes[0], "pickup", "#1565C0"),
    (axes[1], "dropoff", "#C62828"),
]:
    ax.fill_between(
        demand_interval["hour_start"],
        demand_interval[f"{flow}_p05"],
        demand_interval[f"{flow}_p95"],
        color=color, alpha=0.18, label="P05–P95 from 20,000 generated worlds",
    )
    ax.plot(
        demand_interval["hour_start"],
        demand_interval[f"{flow}_p50"],
        color=color, linewidth=2.2, label="P50 generated demand",
    )
    ax.axvline(SIMULATION_KICKOFF, color="black", linestyle=":", label="Kickoff")
    ax.axvline(SIMULATION_KICKOFF.normalize() + pd.Timedelta(days=1), color="#616161", linestyle="--", label="Control day starts")
    ax.set_ylabel(f"{flow.title()} trips / {flow}订单")
    ax.legend(loc="upper right")
axes[0].set_title("20,000 generated two-day demand scenarios / 20,000次连续两天需求模拟")
axes[1].set_xlabel("Local time / 本地时间")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "04_generated_demand_20000_interval.png", dpi=180, bbox_inches="tight")
plt.show()

demand_interval.to_csv(OUTPUT_DIR / "generated_demand_20000_interval.csv", index=False)
display(demand_interval.head())

In [ ]:
# Cell 19 - Business distributions and incremental policy effects / 经营分布与策略增量影响

overall = simulation_summary[simulation_summary["period"] == "overall"].copy()
standard_overall = overall[overall["fleet_name"] == "standard"].copy()

business_quantiles = (
    simulation_summary
    .groupby(["fleet_name", "strategy_name", "period"])
    .agg(
        simulations=("run_id", "nunique"),
        base_request_mean=("base_requests", "mean"),
        retained_request_mean=("total_requests", "mean"),
        price_suppressed_mean=("price_suppressed_requests", "mean"),
        completed_mean=("completed_trips", "mean"),
        unserved_mean=("unserved_trips", "mean"),
        service_rate_mean=("service_rate", "mean"),
        base_fulfillment_mean=("base_demand_fulfillment_rate", "mean"),
        lost_demand_rate_mean=("lost_demand_rate", "mean"),
        stockout_probability=("stockout_flag", "mean"),
        severe_shortage_probability=("severe_shortage_flag", "mean"),
        mean_price_multiplier=("mean_price_multiplier", "mean"),
        max_price_multiplier_p95=("max_price_multiplier", lambda values: values.quantile(0.95)),
        rider_spend_mean=("rider_spend", "mean"),
        profit_p05=("contribution_profit", lambda values: values.quantile(0.05)),
        profit_p50=("contribution_profit", "median"),
        profit_mean=("contribution_profit", "mean"),
        profit_p95=("contribution_profit", lambda values: values.quantile(0.95)),
        revenue_p50=("platform_revenue", "median"),
    )
    .reset_index()
)
business_quantiles.to_csv(OUTPUT_DIR / "business_result_quantiles.csv", index=False)
display(business_quantiles[business_quantiles["period"] == "overall"])

# Compare each surge policy with fixed pricing in the same run and fleet. / 在同一次随机世界、同一车队下，将动态定价与固定价格逐一比较。
increment_source = simulation_summary[
    simulation_summary["strategy_name"].isin(["fixed_price", "balanced_surge", "aggressive_surge"])
].copy()
fixed_reference = (
    increment_source[increment_source["strategy_name"] == "fixed_price"]
    .set_index(["run_id", "fleet_name", "period"])[
        ["contribution_profit", "rider_spend", "completed_trips", "unserved_trips", "price_suppressed_requests"]
    ]
    .add_suffix("_fixed")
)
strategy_incremental = (
    increment_source[increment_source["strategy_name"] != "fixed_price"]
    .join(fixed_reference, on=["run_id", "fleet_name", "period"])
)
for metric in [
    "contribution_profit", "rider_spend", "completed_trips",
    "unserved_trips", "price_suppressed_requests",
]:
    strategy_incremental[f"incremental_{metric}"] = (
        strategy_incremental[metric] - strategy_incremental[f"{metric}_fixed"]
    )

strategy_incremental_comparison = (
    strategy_incremental
    .groupby(["fleet_name", "strategy_name", "period"], as_index=False)
    .agg(
        incremental_profit_mean=("incremental_contribution_profit", "mean"),
        incremental_profit_p05=("incremental_contribution_profit", lambda values: values.quantile(0.05)),
        incremental_profit_p95=("incremental_contribution_profit", lambda values: values.quantile(0.95)),
        incremental_rider_spend_mean=("incremental_rider_spend", "mean"),
        incremental_completed_mean=("incremental_completed_trips", "mean"),
        incremental_unserved_mean=("incremental_unserved_trips", "mean"),
        incremental_price_suppressed_mean=("incremental_price_suppressed_requests", "mean"),
    )
)
strategy_incremental_comparison.to_csv(
    OUTPUT_DIR / "strategy_incremental_vs_fixed.csv", index=False
)
display(strategy_incremental_comparison[
    strategy_incremental_comparison["period"] == "overall"
])

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
profit_plot_lower = standard_overall["contribution_profit"].quantile(0.001)
profit_plot_upper = standard_overall["contribution_profit"].quantile(0.999)
standard_profit_plot = standard_overall[
    standard_overall["contribution_profit"].between(
        profit_plot_lower, profit_plot_upper
    )
].copy()
sns.violinplot(
    data=standard_profit_plot,
    x="strategy_name", y="contribution_profit", hue="strategy_name",
    inner="quartile", cut=0, palette="Set2", legend=False, ax=axes[0],
)
axes[0].set_title("Contribution profit, central 99.8% — standard fleet / 贡献利润中间99.8%—标准车队")
axes[0].set_xlabel("Pricing strategy / 定价策略")
axes[0].set_ylabel("Two-day contribution profit ($) / 两天贡献利润")
axes[0].tick_params(axis="x", rotation=15)

profit_table = business_quantiles[
    (business_quantiles["fleet_name"] == "standard")
    & (business_quantiles["period"] == "overall")
].copy()
for row in profit_table.itertuples(index=False):
    axes[1].plot(
        [row.profit_p05, row.profit_p95],
        [row.strategy_name, row.strategy_name],
        linewidth=7, alpha=0.35,
    )
    axes[1].scatter(row.profit_p50, row.strategy_name, s=90, color="black", zorder=3)
    axes[1].scatter(row.profit_mean, row.strategy_name, s=80, marker="D", color="#D32F2F", zorder=3)
axes[1].set_title("P05–P95, P50 and mean / P05–P95、中位数与平均值")
axes[1].set_xlabel("Two-day contribution profit ($) / 两天贡献利润")
axes[1].set_ylabel("")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "05_profit_distribution_and_interval.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 20 - Vehicle-shortage risk and policy trade-offs / 车辆不足风险与策略取舍

overall_quantiles = business_quantiles[business_quantiles["period"] == "overall"].copy()
fleet_order = [name for name in ["constrained", "lean", "standard", "ample"] if name in FLEET_SCENARIOS]
strategy_order = [name for name in PRICING_STRATEGIES]

stockout_pivot = overall_quantiles.pivot(
    index="fleet_name", columns="strategy_name", values="stockout_probability"
).reindex(index=fleet_order, columns=strategy_order)
severe_pivot = overall_quantiles.pivot(
    index="fleet_name", columns="strategy_name", values="severe_shortage_probability"
).reindex(index=fleet_order, columns=strategy_order)

fig, axes = plt.subplots(1, 3, figsize=(23, 6))
sns.heatmap(
    stockout_pivot * 100,
    annot=True, fmt=".1f", cmap="YlOrRd", vmin=0,
    cbar_kws={"label": "Any-shortage probability (%)"},
    ax=axes[0],
)
axes[0].set_title("Probability of any unserved request / 出现任意未满足请求的概率")
axes[0].set_xlabel("Pricing strategy / 定价策略")
axes[0].set_ylabel("Fleet assumption / 车队假设")

sns.heatmap(
    severe_pivot * 100,
    annot=True, fmt=".1f", cmap="YlOrRd", vmin=0,
    cbar_kws={"label": "Severe-shortage probability (%)"},
    ax=axes[1],
)
axes[1].set_title("Probability lost service reaches 5% / 未服务请求达到原始需求5%的概率")
axes[1].set_xlabel("Pricing strategy / 定价策略")
axes[1].set_ylabel("")

for strategy_name, group in overall_quantiles.groupby("strategy_name"):
    axes[2].scatter(
        group["base_fulfillment_mean"] * 100,
        group["profit_mean"],
        s=120,
        label=strategy_name,
    )
    for row in group.itertuples(index=False):
        axes[2].annotate(
            row.fleet_name,
            (row.base_fulfillment_mean * 100, row.profit_mean),
            xytext=(5, 5), textcoords="offset points", fontsize=8,
        )
axes[2].set_xlabel("Original-demand fulfillment (%) / 原始需求满足率")
axes[2].set_ylabel("Mean two-day contribution profit ($) / 平均两天贡献利润")
axes[2].set_title("Profit versus original-demand fulfillment / 收益与原始需求满足率")
axes[2].legend(title="Strategy / 策略")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "06_vehicle_shortage_and_policy_tradeoff.png", dpi=180, bbox_inches="tight")
plt.show()

display(overall_quantiles.sort_values(
    ["base_fulfillment_mean", "profit_mean"], ascending=False
))

In [ ]:
# Cell 21 - Materialize full synthetic orders for representative runs / 为代表性场景生成完整虚拟订单

balanced_standard = overall[
    (overall["fleet_name"] == "standard")
    & (overall["strategy_name"] == "balanced_surge")
].copy()
target_quantiles = {"low_p05": 0.05, "median_p50": 0.50, "high_p95": 0.95}
representative_runs = []
for label, quantile in target_quantiles.items():
    target_profit = balanced_standard["contribution_profit"].quantile(quantile)
    selected_row = balanced_standard.loc[
        (balanced_standard["contribution_profit"] - target_profit).abs().idxmin()
    ]
    representative_runs.append({
        "label": label,
        "run_id": int(selected_row["run_id"]),
        "target_quantile": quantile,
        "contribution_profit": selected_row["contribution_profit"],
    })
representative_runs = pd.DataFrame(representative_runs)
display(representative_runs)

def lognormal_parameters(mean, mean_square, default_cv=0.45):
    """Convert mean and standard deviation to lognormal parameters. / 将均值和标准差转换为对数正态参数。"""
    # Convert mean and second moment to lognormal parameters. / 将均值和二阶矩转换为对数正态参数。
    mean = max(float(np.nan_to_num(mean, nan=1.0)), 1e-3)
    variance = max(
        float(np.nan_to_num(mean_square, nan=mean ** 2 * (1 + default_cv ** 2))) - mean ** 2,
        0,
    )
    if variance <= 1e-8:
        variance = (mean * default_cv) ** 2
    sigma2 = np.log1p(variance / (mean ** 2))
    return np.log(mean) - 0.5 * sigma2, np.sqrt(sigma2)


route_groups = {
    h3_cell: group.reset_index(drop=True)
    for h3_cell, group in route_profile.groupby("pickup_h3")
}

def materialize_synthetic_orders(run_id, label):
    """Expand one representative aggregate run into synthetic trip records. / 将一个代表性聚合场景展开为合成行程记录。"""
    # Materialize every base request, including requests suppressed by price. / 生成每一个原始请求，包括因价格上涨而退出的请求。
    demand = generate_demand_scenario(run_id)
    config = next(
        config for config in CONFIGURATIONS
        if config["fleet_name"] == "standard"
        and config["strategy_name"] == "balanced_surge"
    )
    _, detail = run_operating_policy(demand, config, return_cell_detail=True)
    rng = np.random.default_rng(int(demand["scenario_seed"]) + 99173)
    order_frames = []
    supply_frames = []

    for row in detail.itertuples(index=False):
        n_base_requests = int(row.base_requests)
        n_effective = int(row.effective_requests)
        if n_base_requests > 0:
            routes = route_groups.get(row.h3)
            if routes is None or routes.empty:
                routes = route_profile
            probabilities = routes["n"].to_numpy(dtype=float)
            probabilities = probabilities / probabilities.sum()
            route_indices = rng.choice(len(routes), size=n_base_requests, p=probabilities)
            sampled_routes = routes.iloc[route_indices].reset_index(drop=True)
            request_times = pd.Timestamp(row.hour_start) + pd.to_timedelta(
                rng.uniform(0, 3600, size=n_base_requests), unit="s"
            )

            accepted_flags = np.zeros(n_base_requests, dtype=int)
            accepted_flags[:n_effective] = 1
            rng.shuffle(accepted_flags)
            accepted_indices = np.flatnonzero(accepted_flags)
            served_flags = np.zeros(n_base_requests, dtype=int)
            if len(accepted_indices):
                served_indices = rng.choice(
                    accepted_indices,
                    size=min(int(row.completed), len(accepted_indices)),
                    replace=False,
                )
                served_flags[served_indices] = 1

            miles = np.zeros(n_base_requests)
            seconds = np.zeros(n_base_requests)
            base_fare = np.zeros(n_base_requests)
            for index, route in sampled_routes.iterrows():
                mu, sigma = lognormal_parameters(route["avg_miles"], route["mean_sq_miles"], 0.40)
                miles[index] = rng.lognormal(mu, sigma)
                mu, sigma = lognormal_parameters(route["avg_seconds"], route["mean_sq_seconds"], 0.35)
                seconds[index] = rng.lognormal(mu, sigma)
                mu, sigma = lognormal_parameters(route["avg_fare"], route["mean_sq_fare"], 0.35)
                base_fare[index] = rng.lognormal(mu, sigma)

            econ_key = (row.h3, pd.Timestamp(row.hour_start).hour)
            econ = economics_lookup.loc[econ_key] if econ_key in economics_lookup.index else None
            tip_rate = float(econ["recorded_tip_rate"]) if econ is not None else 0.20
            positive_tip_mean = float(np.nan_to_num(econ["avg_positive_tip"], nan=3.0)) if econ is not None else 3.0
            has_tip = rng.binomial(1, np.clip(tip_rate, 0, 1), size=n_base_requests)
            tips = has_tip * rng.gamma(
                shape=2.0,
                scale=max(positive_tip_mean, 0.1) / 2.0,
                size=n_base_requests,
            )
            charged_fare = base_fare * row.price_multiplier * served_flags
            tips = tips * served_flags

            order_frames.append(pd.DataFrame({
                "simulation_label": label,
                "run_id": run_id,
                "scenario_seed": demand["scenario_seed"],
                "request_time": request_times,
                "pickup_h3": row.h3,
                "dropoff_h3": sampled_routes["dropoff_h3"].to_numpy(),
                "trip_miles": np.clip(miles, 0, 100),
                "trip_seconds": np.clip(seconds, 0, 21600),
                "accepted_after_price": accepted_flags,
                "price_suppressed": 1 - accepted_flags,
                "served": served_flags,
                "price_multiplier": row.price_multiplier,
                "base_fare": base_fare,
                "charged_fare": charged_fare,
                "tip": tips,
                "rider_payment": charged_fare + tips,
                "day_type": row.day_type,
                "synthetic_data": True,
            }))

        n_supply = int(row.dropoff_arrivals + row.extra_supply)
        if n_supply > 0:
            supply_frames.append(pd.DataFrame({
                "simulation_label": label,
                "run_id": run_id,
                "arrival_time": pd.Timestamp(row.hour_start) + pd.to_timedelta(
                    rng.uniform(0, 3600, size=n_supply), unit="s"
                ),
                "dropoff_h3": row.h3,
                "supply_event_type": "generated_available_inbound_vehicle",
                "synthetic_data": True,
            }))

    orders = pd.concat(order_frames, ignore_index=True) if order_frames else pd.DataFrame()
    supply = pd.concat(supply_frames, ignore_index=True) if supply_frames else pd.DataFrame()
    orders.to_csv(
        TRIP_EXPORT_DIR / f"synthetic_orders_{label}_run_{run_id}.csv.gz",
        index=False, compression="gzip",
    )
    supply.to_csv(
        TRIP_EXPORT_DIR / f"synthetic_inbound_supply_{label}_run_{run_id}.csv.gz",
        index=False, compression="gzip",
    )
    return orders, supply


representative_export_summary = []
for scenario in representative_runs.itertuples(index=False):
    orders, supply = materialize_synthetic_orders(scenario.run_id, scenario.label)
    representative_export_summary.append({
        "label": scenario.label,
        "run_id": scenario.run_id,
        "base_requests": len(orders),
        "accepted_after_price": int(orders["accepted_after_price"].sum()) if len(orders) else 0,
        "price_suppressed": int(orders["price_suppressed"].sum()) if len(orders) else 0,
        "served_orders": int(orders["served"].sum()) if len(orders) else 0,
        "inbound_supply_events": len(supply),
        "order_file": str(TRIP_EXPORT_DIR / f"synthetic_orders_{scenario.label}_run_{scenario.run_id}.csv.gz"),
    })
representative_export_summary = pd.DataFrame(representative_export_summary)
representative_export_summary.to_csv(
    OUTPUT_DIR / "representative_synthetic_trip_exports.csv", index=False
)
display(representative_export_summary)

In [ ]:
# Cell 22 - Final validation and output inventory / 最终检查与输出清单

overall_standard_balanced = simulation_summary[
    (simulation_summary["period"] == "overall")
    & (simulation_summary["fleet_name"] == "standard")
    & (simulation_summary["strategy_name"] == "balanced_surge")
]

event_curve_has_signal = bool(
    max(
        np.exp(zone_template["pickup_log_excess"]).max(),
        np.exp(zone_template["dropoff_log_excess"]).max(),
    ) > 1.05
)
policy_count = overall[["fleet_name", "strategy_name"]].drop_duplicates().shape[0]
selected_activity_valid = bool(
    (selected_metadata["total_activity"] >= MIN_H3_TOTAL_ACTIVITY).all()
)

# CSV round-trips can move a floating-point boundary by a tiny amount. / CSV 往返读写可能让浮点边界产生极小误差。
FLOAT_TOLERANCE = 1e-9
event_strength_observed_min = float(simulation_summary["event_strength"].min())
event_strength_observed_max = float(simulation_summary["event_strength"].max())
event_strength_valid = bool(
    np.isfinite(simulation_summary["event_strength"]).all()
    and event_strength_observed_min >= EVENT_STRENGTH_MIN - FLOAT_TOLERANCE
    and event_strength_observed_max <= EVENT_STRENGTH_MAX + FLOAT_TOLERANCE
)

count_columns = [
    "base_requests", "total_requests", "completed_trips", "lost_demand",
]
count_values = simulation_summary[count_columns].to_numpy(dtype=float)
nonnegative_counts_valid = bool(
    np.isfinite(count_values).all()
    and (count_values >= -FLOAT_TOLERANCE).all()
)
request_accounting_error = (
    simulation_summary["price_suppressed_requests"].to_numpy(dtype=float)
    + simulation_summary["total_requests"].to_numpy(dtype=float)
    - simulation_summary["base_requests"].to_numpy(dtype=float)
)
request_accounting_valid = bool(
    np.isfinite(request_accounting_error).all()
    and np.allclose(request_accounting_error, 0.0, atol=FLOAT_TOLERANCE, rtol=0.0)
)

service_values = simulation_summary[
    ["service_rate", "base_demand_fulfillment_rate"]
].to_numpy(dtype=float)
service_metrics_valid = bool(
    np.isfinite(service_values).all()
    and (service_values >= -FLOAT_TOLERANCE).all()
    and (service_values <= 1.0 + FLOAT_TOLERANCE).all()
)

expected_representative_labels = {"low_p05", "median_p50", "high_p95"}
representative_labels = set(representative_export_summary["label"].astype(str))
representative_files_valid = bool(
    representative_labels == expected_representative_labels
    and representative_export_summary["order_file"].map(
        lambda value: Path(value).exists()
    ).all()
)

final_checks = pd.Series({
    "MatrixOne source is nonempty / MatrixOne源表非空": int(source_check.loc[0, "n"]) > 0,
    "17 training games / 17场训练比赛": len(train_games) == 17,
    "8 held-out 2024 games / 8场2024验证比赛": len(validation_games) == 8,
    "all selected H3 are active / 最终H3均有真实活动": selected_activity_valid,
    "event curve is non-flat / 事件曲线不是平线": event_curve_has_signal,
    "event strength stays near history / 事件强度保持在历史范围附近": event_strength_valid,
    "20,000 runs completed / 完成20,000次模拟": simulation_summary["run_id"].nunique() == N_SIMULATIONS,
    "all policy combinations / 所有策略组合": policy_count == len(CONFIGURATIONS),
    "nonnegative counts / 数量非负": nonnegative_counts_valid,
    "request accounting valid / 请求数量守恒": request_accounting_valid,
    "service metrics valid / 服务指标有效": service_metrics_valid,
    "representative trip files created / 已生成代表性逐笔订单": representative_files_valid,
    "business outputs finite / 经营结果为有限数值": bool(
        np.isfinite(overall_standard_balanced["contribution_profit"]).all()
    ),
})
display(final_checks.to_frame("passed"))

failed_checks = final_checks.index[~final_checks].tolist()
if failed_checks:
    print("Failed checks / 未通过项目:")
    for item in failed_checks:
        print("-", item)

validation_details = pd.Series({
    "event_strength_expected_min": EVENT_STRENGTH_MIN,
    "event_strength_observed_min": event_strength_observed_min,
    "event_strength_expected_max": EVENT_STRENGTH_MAX,
    "event_strength_observed_max": event_strength_observed_max,
    "maximum_request_accounting_error": float(np.max(np.abs(request_accounting_error))),
    "service_metric_minimum": float(np.min(service_values)),
    "service_metric_maximum": float(np.max(service_values)),
    "completed_run_count": int(simulation_summary["run_id"].nunique()),
    "policy_combination_count": int(policy_count),
    "representative_labels": ", ".join(sorted(representative_labels)),
})
display(validation_details.to_frame("value"))

diagnostic_summary = pd.Series({
    "stockout_probability_min": overall_quantiles["stockout_probability"].min(),
    "stockout_probability_max": overall_quantiles["stockout_probability"].max(),
    "severe_shortage_probability_min": overall_quantiles["severe_shortage_probability"].min(),
    "severe_shortage_probability_max": overall_quantiles["severe_shortage_probability"].max(),
    "base_fulfillment_min": overall_quantiles["base_fulfillment_mean"].min(),
    "base_fulfillment_max": overall_quantiles["base_fulfillment_mean"].max(),
    "maximum_mean_price_multiplier": overall_quantiles["mean_price_multiplier"].max(),
    "standard_fleet_profit_spread": (
        standard_overall.groupby("strategy_name")["contribution_profit"].mean().max()
        - standard_overall.groupby("strategy_name")["contribution_profit"].mean().min()
    ),
    "base_request_median": overall["base_requests"].median(),
    "base_request_p999": overall["base_requests"].quantile(0.999),
    "base_request_maximum": overall["base_requests"].max(),
    "base_request_max_to_median": (
        overall["base_requests"].max()
        / max(overall["base_requests"].median(), 1)
    ),
})
display(diagnostic_summary.to_frame("value"))

if diagnostic_summary["stockout_probability_max"] == 0:
    print("Warning: no stockout scenario was observed; fleet stress is still too weak. / 警告：没有出现缺车，车队压力仍然过弱。")
if diagnostic_summary["maximum_mean_price_multiplier"] <= 1.0001:
    print("Warning: surge pricing was never activated. / 警告：动态定价从未触发。")
if diagnostic_summary["base_request_max_to_median"] > 4:
    print("Warning: generated demand tail remains unusually large. / 警告：生成需求的极端尾部仍然偏大。")

output_inventory = pd.DataFrame({
    "artifact / 产物": [
        "H3 selection metadata", "Static real map", "Interactive real map",
        "2024 demand validation", "20,000-run business summary",
        "Demand simulation interval", "Business quantiles",
        "Incremental strategy comparison", "Selected-H3 event sensitivity",
        "Representative synthetic trips",
    ],
    "path / 路径": [
        OUTPUT_DIR / "selected_h3_metadata.csv",
        MAP_DIR / "01_soldier_field_h3_selection_real_map.png",
        MAP_DIR / "soldier_field_h3_selection_interactive.html",
        OUTPUT_DIR / "demand_generator_2024_metrics.csv",
        OUTPUT_DIR / "monte_carlo_20000_business_summary.csv.gz",
        OUTPUT_DIR / "generated_demand_20000_interval.csv",
        OUTPUT_DIR / "business_result_quantiles.csv",
        OUTPUT_DIR / "strategy_incremental_vs_fixed.csv",
        OUTPUT_DIR / "selected_h3_event_sensitivity.csv",
        TRIP_EXPORT_DIR,
    ],
})
display(output_inventory)

if failed_checks:
    raise RuntimeError(
        "Final validation failed for: "
        + "; ".join(failed_checks)
        + " / 最终检查未通过，请查看上方具体项目。"
    )

print("\nDONE / 完成")
print("This is a scenario simulator, not audited company finance. / 这是情景模拟器，不是经审计的公司财务结果。")
print("Vehicle, elasticity, take-rate, incentive, and cost inputs remain explicit assumptions. / 车辆、弹性、抽成、奖励和成本仍是明确假设。")

## Result interpretation / 结果解释

- **P05–P95** is calculated from 20,000 complete generated worlds. Each world first generates hourly pickup and dropoff demand, local supply availability, and vehicle retention; the business metrics are then calculated from those generated records. / **P05–P95** 来自20,000个完整随机世界。每个世界先生成逐小时 pickup、dropoff、本地可用供给和车辆留存，再从这些生成记录中计算经营指标。
- **Base demand** is generated demand before pricing. **Retained requests** are requests that remain after the assumed price response. **Completed trips** are retained requests with a vehicle. This distinction prevents price-suppressed demand from being presented as improved service. / **原始需求**是定价前生成的需求；**保留请求**是考虑价格反应后仍继续叫车的请求；**完成订单**是其中有车可服务的部分。这样不会把被价格压掉的需求误写成服务改善。
- **Operational service rate** is completed trips divided by retained requests. **Original-demand fulfillment** is completed trips divided by base demand and is the stricter customer-impact metric. / **运营完成率**是完成订单除以保留请求；**原始需求满足率**是完成订单除以原始需求，后者对乘客影响的衡量更严格。
- **Contribution profit** equals assumed platform revenue minus assumed driver incentives and per-trip variable cost. It is not audited net profit. / **贡献利润**等于假设的平台收入减去假设的司机奖励和单笔变动成本，不是经审计的净利润。
- **Stockout probability** is the percentage of generated worlds with at least one unserved retained request. **Severe shortage probability** is the percentage where unserved requests reach at least 5% of original demand. Neither reveals real Chicago driver supply. / **缺车概率**是至少出现一个未服务保留请求的随机世界比例；**严重缺车概率**是未服务请求达到原始需求5%的随机世界比例。两者都不代表芝加哥真实司机供给。
- The best policy is not selected from profit alone. Original-demand fulfillment, shortage probability, price multiplier, rider spend, and incremental results versus fixed pricing must be reviewed together. / 不能只根据利润选择策略，还要同时看原始需求满足率、缺车概率、价格倍数、乘客支付，以及相对固定价格的增量结果。

## Result snapshot / 结果快照

- The demand generator is validated on 8 unseen 2024 games before business simulation. Pickup WAPE is 19.15% and dropoff WAPE is 22.78%. / 需求生成器在业务模拟前用 8 场未参与训练的 2024 比赛验证，pickup WAPE 为 19.15%，dropoff WAPE 为 22.78%。
- The final experiment runs 20,000 two-day scenarios over four fleet sizes and three pricing policies. / 最终实验在四种车队规模和三种定价策略下运行 20,000 个连续两天场景。
- Under the standard-fleet assumptions, balanced pricing produces the highest mean contribution profit, while aggressive pricing lowers severe shortage risk more strongly. / 在标准车队假设下，平衡定价的平均贡献利润最高，激进定价对严重缺车风险的降低更明显。

This is the end of the current mainline. The exported tables support client review and later policy changes without rebuilding the raw trip dataset.  
这里是当前主线的终点。导出的表格可以支持客户复核和后续策略调整，不需要重新构建原始订单数据。